# PunterStat — Unified Football ML & Dixon-Coles Notebook (PS-05 + LightGBM + Supabase)

100% **Cloudflare-free** unified pipeline combining **PS-05's research-grade parametric Dixon-Coles MLE** with **Dual LightGBM Poisson + 1X2 models**, backed directly by **Supabase Postgres** across all **16 European leagues** (`2005/06 → 2025/26`).

### Kaggle Setup Checklist
1. **Internet:** Turn **ON** in the right-hand Kaggle Notebook Settings panel.
2. **Accelerator:** CPU is plenty fast (~2–3 min for 133k+ matches); GPU is optional.
3. **Secret:** Go to **Add-ons → Secrets**, ensure **`SUPABASE_DB_URL`** is toggled **ON**, and click **Save Version → Save & Run All (Commit)**.

### Unified Pipeline (PS-05 + LightGBM)
1. **Direct Supabase Seed & Load (Zero Cloudflare):** Auto-seeds `public.historical_matches` across all 16 leagues in parallel if empty, then loads straight from Supabase.
2. **PS-05 Data Quality & Low-Score Diagnostics:** Verifies schema integrity, odds coverage, and empirical `0-0 / 1-0 / 0-1 / 1-1` correlation vs independent Poisson.
3. **Leakage-Free Feature Engineering:** Chronological Elo ratings + shifted rolling form (`5 / 10 / 20` games + venue-specific form + rest days).
4. **PS-05 Parametric Time-Decayed Dixon-Coles MLE:** Fits per-league `(attack, defense, home_adv, rho)` with chronological $\xi$ time-decay tuning + out-of-fold temperature calibration.
5. **Dual LightGBM Poisson + 3-Class 1X2 + Hybrid Ensemble:** Trains pure-stats LightGBM Poisson, 3-class 1X2, a **Hybrid PS-05 + LightGBM ensemble**, and a **Stats + Market Blend**.
6. **PS-05 Clustered Bootstrap CIs & Exports:** Matchday-clustered 95% confidence intervals, PunterStat 5-bucket reliability curves, and lightweight `dixon_coles_mle_params.json` + `model_metadata.json` for Vercel/Supabase.

In [ ]:
# Install psycopg2-binary (direct Supabase Postgres) and ML libraries
!pip -q install psycopg2-binary lightgbm scipy scikit-learn pyarrow

import concurrent.futures
import datetime as dt_mod
import hashlib
import io
import json
import math
import os
import re
import time
import urllib.parse
import urllib.request
import warnings
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize, minimize_scalar
from scipy.special import softmax
from scipy.stats import poisson

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

# ── Model Registry & Pipeline Config (v2.1, 56 Leagues, 100% Supabase, Zero Cloudflare) ───
MODEL_VERSION = "2.1.0"
FEATURE_SCHEMA_VERSION = "2.1"

# Automatically seeds public.historical_matches from football-data.co.uk if empty.
SEED_SUPABASE_IF_EMPTY = True
# Set True whenever you want to refresh the current season (or all seasons) in Supabase.
FORCE_REFRESH_SUPABASE = False

# All 22 European leagues from football-data.co.uk/mmz4281/ (100% of FDCO Europe)
LEAGUES = {
    "E0": "Premier League",
    "E1": "Championship",
    "E2": "League One",
    "E3": "League Two",
    "EC": "National League",
    "SP1": "La Liga",
    "SP2": "Segunda División",
    "D1": "Bundesliga",
    "D2": "2. Bundesliga",
    "I1": "Serie A",
    "I2": "Serie B",
    "F1": "Ligue 1",
    "F2": "Ligue 2",
    "N1": "Eredivisie",
    "B1": "First Division A",
    "P1": "Primeira Liga",
    "T1": "Süper Lig",
    "SC0": "Scottish Premiership",
    "SC1": "Scottish Championship",
    "SC2": "Scottish League One",
    "SC3": "Scottish League Two",
    "G1": "Super League",
}

# All 16 global "Extra" leagues from football-data.co.uk/new/{CODE}.csv (100% of FDCO Global)
EXTRA_LEAGUES = {
    "ARG": "Liga Profesional (Argentina)",
    "AUT": "Austrian Bundesliga",
    "BRA": "Brasileirão Série A",
    "CHN": "Chinese Super League",
    "DNK": "Danish Superliga",
    "FIN": "Veikkausliiga (Finland)",
    "IRL": "League of Ireland",
    "JPN": "J1 League (Japan)",
    "MEX": "Liga MX (Mexico)",
    "NOR": "Eliteserien (Norway)",
    "POL": "Ekstraklasa (Poland)",
    "ROU": "Liga I (Romania)",
    "RUS": "Russian Premier League",
    "SWE": "Allsvenskan (Sweden)",
    "SWZ": "Swiss Super League",
    "USA": "MLS (USA)",
}

# 18 additional African, Continental & Global leagues via SportsAPIPro v2
# Format: code -> (display_name, search_query, canonical_tournament_id_or_None)
SPORTSAPIPRO_LEAGUES = {
    # African Domestic Leagues (9)
    "NG1":   ("Nigeria Premier Football League", "Nigeria Premier Football League", None),
    "DZ1":   ("Algerian Ligue 1",                "Algeria Ligue 1",                 None),
    "EG1":   ("Egyptian Premier League",         "Egyptian Premier League",         None),
    "SN1":   ("Senegal Ligue 1",                 "Senegal Ligue 1",                 None),
    "MA1":   ("Botola Pro (Morocco)",            "Botola Pro",                      None),
    "TN1":   ("Tunisian Ligue 1",                "Ligue 1 Tunisia",                 None),
    "ZA1":   ("South Africa Premiership (PSL)",  "Premiership South Africa",        None),
    "GH1":   ("Ghana Premier League",            "Ghana Premier League",            None),
    "KE1":   ("FKF Premier League (Kenya)",      "Premier League Kenya",            None),
    # Continental Club Competitions (5)
    "UCL":   ("UEFA Champions League",           "UEFA Champions League",           7),
    "UEL":   ("UEFA Europa League",              "UEFA Europa League",              679),
    "UECL":  ("UEFA Conference League",          "UEFA Conference League",          None),
    "CAFCL": ("CAF Champions League",            "CAF Champions League",            None),
    "LIB":   ("Copa Libertadores",               "CONMEBOL Libertadores",           None),
    # Global Top Flights not on FDCO (4)
    "SAU":   ("Saudi Pro League",                "Saudi Pro League",                None),
    "AUS":   ("A-League Men (Australia)",        "A-League Men",                    None),
    "KOR":   ("K League 1 (South Korea)",        "K League 1",                      None),
    "COL":   ("Categoría Primera A (Colombia)",  "Primera A Colombia",              None),
}

ENABLE_EXTRA_LEAGUES = True
ENABLE_INTERNATIONAL_SEED = True   # martj42 Kaggle dataset (AFCON, World Cup, international matches)
ENABLE_SPORTSAPIPRO_LEAGUES = True # 18 African, Continental & Global leagues when SPORTSAPIPRO_API_KEY is set
SPORTSAPIPRO_MAX_REQUESTS = 80     # Stay safely under 100 req/day free quota

# Seasons from 2005/06 -> 2026/27 (22 seasons; strong shots/corners/odds coverage)
SEED_SEASONS = [
    f"{str(y)[2:]}{str(y + 1)[2:]}"
    for y in range(2005, 2027)
]

# Nested chronological split (eliminates validation-set overfitting for calibration!)
MIN_MATCH_DATE = "2005-08-01"
TRAIN_END_DATE = "2022-07-01"  # train: <= 2021/22
TUNE_END_DATE  = "2023-07-01"  # tune:  2022/23 (early stopping + xi decay tuning)
CAL_END_DATE   = "2024-07-01"  # cal:   2023/24 (temperature + per-league rho + hybrid alpha + bucket cal)
VAL_END_DATE   = CAL_END_DATE  # test:  2024/25+ (strictly untouched held-out evaluation)

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./working")
WORK_DIR.mkdir(parents=True, exist_ok=True)
CACHE_PARQUET = WORK_DIR / "punterstat_matches.parquet"

print(
    f"PunterStat ML v{MODEL_VERSION} (schema {FEATURE_SCHEMA_VERSION}) | "
    f"FDCO Europe: {len(LEAGUES)} | FDCO Extra: {len(EXTRA_LEAGUES)} | "
    f"SportsAPIPro: {len(SPORTSAPIPRO_LEAGUES)} | Total Domestic/Cont: {len(LEAGUES) + len(EXTRA_LEAGUES) + len(SPORTSAPIPRO_LEAGUES)} | "
    f"Seasons: {SEED_SEASONS[0]}..{SEED_SEASONS[-1]}"
)

## 1. Direct Supabase Seed & Load (Zero Cloudflare)
- Connects directly to **Supabase Postgres** via `SUPABASE_DB_URL` (`sslmode=require`).
- If `public.historical_matches` is empty (or `FORCE_REFRESH_SUPABASE = True`), downloads all **16 leagues** (`2005/06 → 2025/26`) in parallel from `football-data.co.uk` and bulk-upserts them straight into Supabase (`ON CONFLICT (external_id) DO UPDATE`).
- Then loads the full dataset directly from Supabase for feature engineering and training.

In [ ]:
def get_secret(name: str) -> str | None:
    """Read a secret from Kaggle Secrets first, then fall back to os.environ."""
    try:
        from kaggle_secrets import UserSecretsClient
        val = UserSecretsClient().get_secret(name)
        if val:
            return val
    except Exception:
        pass
    return os.environ.get(name)


def sanitize_pg_uri(uri: str) -> str:
    """Safely percent-encode special characters (like # or @) in the password part of a Postgres URI."""
    uri = uri.strip()
    m = re.match(r"^(postgres(?:ql)?://)([^:]+):(.+)@([^/@]+(?::\d+)?/.*)$", uri)
    if not m:
        return uri
    scheme, user, raw_pass, host_and_db = m.groups()
    encoded_pass = urllib.parse.quote(urllib.parse.unquote(raw_pass), safe="")
    return f"{scheme}{user}:{encoded_pass}@{host_and_db}"


def slugify(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", str(name).lower()).strip("-")


def season_code_to_label(code: str) -> str:
    # "0506" -> "2005/06", "2425" -> "2024/25"
    yy1 = int(code[:2])
    year1 = 1900 + yy1 if yy1 >= 93 else 2000 + yy1
    return f"{year1}/{code[2:]}"


def parse_fdco_date(series: pd.Series) -> pd.Series:
    """Parse Football-Data.co.uk dd/mm/yy or dd/mm/yyyy dates cleanly."""
    s = series.astype(str).str.strip()
    dt = pd.to_datetime(s, format="%d/%m/%Y", errors="coerce")
    mask = dt.isna()
    if mask.any():
        dt.loc[mask] = pd.to_datetime(s.loc[mask], format="%d/%m/%y", errors="coerce")
    mask = dt.isna()
    if mask.any():
        dt.loc[mask] = pd.to_datetime(s.loc[mask], dayfirst=True, errors="coerce")
    return dt


def first_col(df: pd.DataFrame, candidates: list[str]) -> pd.Series:
    for c in candidates:
        if c in df.columns:
            return pd.to_numeric(df[c], errors="coerce")
    return pd.Series(np.nan, index=df.index, dtype="float64")


def normalize_fdco_df(raw: pd.DataFrame, league_code: str, season_code: str) -> pd.DataFrame:
    if raw.empty or "HomeTeam" not in raw.columns or "AwayTeam" not in raw.columns:
        return pd.DataFrame()

    df = raw.dropna(subset=["HomeTeam", "AwayTeam", "Date"]).copy()
    if df.empty:
        return pd.DataFrame()

    match_dates = parse_fdco_date(df["Date"])
    home_teams = df["HomeTeam"].astype(str).str.strip()
    away_teams = df["AwayTeam"].astype(str).str.strip()

    out = pd.DataFrame({
        "source": "football-data-co-uk",
        "league_code": league_code,
        "league_name": LEAGUES.get(league_code, league_code),
        "season": season_code_to_label(season_code),
        "match_date": match_dates,
        "match_time": df["Time"].astype(str).str.strip() if "Time" in df.columns else None,
        "home_team": home_teams,
        "away_team": away_teams,
        "home_goals": first_col(df, ["FTHG", "HG"]),
        "away_goals": first_col(df, ["FTAG", "AG"]),
        "ht_home_goals": first_col(df, ["HTHG"]),
        "ht_away_goals": first_col(df, ["HTAG"]),
        "home_shots": first_col(df, ["HS"]),
        "away_shots": first_col(df, ["AS"]),
        "home_sot": first_col(df, ["HST"]),
        "away_sot": first_col(df, ["AST"]),
        "home_corners": first_col(df, ["HC"]),
        "away_corners": first_col(df, ["AC"]),
        "home_fouls": first_col(df, ["HF"]),
        "away_fouls": first_col(df, ["AF"]),
        "home_yellow": first_col(df, ["HY"]),
        "away_yellow": first_col(df, ["AY"]),
        "home_red": first_col(df, ["HR"]),
        "away_red": first_col(df, ["AR"]),
        "avg_home_odds": first_col(df, ["AvgH", "BbAvH", "B365H"]),
        "avg_draw_odds": first_col(df, ["AvgD", "BbAvD", "B365D"]),
        "avg_away_odds": first_col(df, ["AvgA", "BbAvA", "B365A"]),
        "max_home_odds": first_col(df, ["MaxH", "BbMxH"]),
        "max_draw_odds": first_col(df, ["MaxD", "BbMxD"]),
        "max_away_odds": first_col(df, ["MaxA", "BbMxA"]),
        "over25_odds": first_col(df, ["Avg>2.5", "AvgO2.5", "BbAv>2.5", "B365>2.5"]),
        "under25_odds": first_col(df, ["Avg<2.5", "AvgU2.5", "BbAv<2.5", "B365<2.5"]),
        "max_over25_odds": first_col(df, ["Max>2.5", "MaxO2.5", "BbMx>2.5"]),
        "max_under25_odds": first_col(df, ["Max<2.5", "MaxU2.5", "BbMx<2.5"]),
        "handi_size": first_col(df, ["AHh", "BbAHh"]),
        "handi_home_odds": first_col(df, ["AvgAHH", "BbAvAHH"]),
        "handi_away_odds": first_col(df, ["AvgAHA", "BbAvAHA"]),
        "ps_home_odds": first_col(df, ["PSCH", "PSH"]),
        "ps_draw_odds": first_col(df, ["PSCD", "PSD"]),
        "ps_away_odds": first_col(df, ["PSCA", "PSA"]),
    })

    out = out.dropna(subset=["match_date", "home_goals", "away_goals"]).copy()
    out["home_goals"] = out["home_goals"].astype(int)
    out["away_goals"] = out["away_goals"].astype(int)
    out["result"] = np.where(
        out["home_goals"] > out["away_goals"], "H",
        np.where(out["home_goals"] < out["away_goals"], "A", "D"),
    )
    out["external_id"] = [
        f"{lc}_{dt.strftime('%Y-%m-%d')}_{slugify(ht)}_{slugify(at)}"
        for lc, dt, ht, at in zip(out["league_code"], out["match_date"], out["home_team"], out["away_team"])
    ]
    return out


def _fetch_one_csv(args: tuple[str, str]) -> pd.DataFrame | None:
    league_code, season_code = args
    url = f"https://www.football-data.co.uk/mmz4281/{season_code}/{league_code}.csv"
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "PunterStat-Kaggle/1.0"})
        with urllib.request.urlopen(req, timeout=15) as resp:
            raw_bytes = resp.read()
        raw_df = pd.read_csv(io.BytesIO(raw_bytes), encoding="latin1", on_bad_lines="skip")
        norm = normalize_fdco_df(raw_df, league_code, season_code)
        return norm if not norm.empty else None
    except Exception:
        return None


def download_fdco_all(leagues: dict[str, str], seasons: list[str]) -> pd.DataFrame:
    tasks = [(lc, sc) for lc in leagues for sc in seasons]
    print(f"Downloading {len(tasks)} CSVs ({len(leagues)} leagues x {len(seasons)} seasons) in parallel...")
    t_dl = time.time()
    frames = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as ex:
        for res in ex.map(_fetch_one_csv, tasks):
            if res is not None:
                frames.append(res)
    df = pd.concat(frames, ignore_index=True)
    df = df.drop_duplicates(subset=["external_id"], keep="last").reset_index(drop=True)
    print(f"Downloaded {len(frames)}/{len(tasks)} CSVs -> {len(df):,} unique matches in {time.time() - t_dl:.1f}s.")
    return df


def _to_int_or_none(v):
    if v is None or (isinstance(v, float) and math.isnan(v)) or pd.isna(v):
        return None
    return int(round(float(v)))


def _to_float_or_none(v):
    if v is None or (isinstance(v, float) and math.isnan(v)) or pd.isna(v):
        return None
    val = float(v)
    return val if math.isfinite(val) else None


def upsert_to_supabase(df: pd.DataFrame, uri: str, batch_size: int = 2000) -> int:
    """Bulk-upsert matches into public.historical_matches and Pinnacle closing odds into public.match_odds."""
    import psycopg2
    from psycopg2.extras import execute_values

    cols = [
        "source", "external_id", "league_code", "league_name", "season",
        "match_date", "match_time", "home_team", "away_team",
        "home_goals", "away_goals", "result",
        "ht_home_goals", "ht_away_goals",
        "home_shots", "away_shots", "home_shots_on_target", "away_shots_on_target",
        "home_corners", "away_corners", "home_fouls", "away_fouls",
        "home_yellow_cards", "away_yellow_cards", "home_red_cards", "away_red_cards",
        "avg_home_odds", "avg_draw_odds", "avg_away_odds",
        "max_home_odds", "max_draw_odds", "max_away_odds",
        "over25_odds", "under25_odds", "max_over25_odds", "max_under25_odds",
        "handi_size", "handi_home_odds", "handi_away_odds",
    ]
    update_cols = [c for c in cols if c != "external_id"]
    update_sql = ", ".join(f"{c} = EXCLUDED.{c}" for c in update_cols) + ", updated_at = NOW()"
    insert_sql = (
        f"INSERT INTO public.historical_matches ({', '.join(cols)}) VALUES %s "
        f"ON CONFLICT (external_id) DO UPDATE SET {update_sql}"
    )

    records = []
    ps_records = []
    for r in df.itertuples(index=False):
        mt = r.match_time if (r.match_time and str(r.match_time).lower() != "nan") else None
        records.append((
            r.source, r.external_id, r.league_code, r.league_name, r.season,
            r.match_date.strftime("%Y-%m-%d"), mt, r.home_team, r.away_team,
            _to_int_or_none(r.home_goals), _to_int_or_none(r.away_goals), r.result,
            _to_int_or_none(r.ht_home_goals), _to_int_or_none(r.ht_away_goals),
            _to_int_or_none(r.home_shots), _to_int_or_none(r.away_shots),
            _to_int_or_none(r.home_sot), _to_int_or_none(r.away_sot),
            _to_int_or_none(r.home_corners), _to_int_or_none(r.away_corners),
            _to_int_or_none(r.home_fouls), _to_int_or_none(r.away_fouls),
            _to_int_or_none(r.home_yellow), _to_int_or_none(r.away_yellow),
            _to_int_or_none(r.home_red), _to_int_or_none(r.away_red),
            _to_float_or_none(r.avg_home_odds), _to_float_or_none(r.avg_draw_odds), _to_float_or_none(r.avg_away_odds),
            _to_float_or_none(r.max_home_odds), _to_float_or_none(r.max_draw_odds), _to_float_or_none(r.max_away_odds),
            _to_float_or_none(r.over25_odds), _to_float_or_none(r.under25_odds),
            _to_float_or_none(r.max_over25_odds), _to_float_or_none(r.max_under25_odds),
            _to_float_or_none(r.handi_size), _to_float_or_none(r.handi_home_odds), _to_float_or_none(r.handi_away_odds),
        ))
        psh = _to_float_or_none(r.ps_home_odds)
        psd = _to_float_or_none(r.ps_draw_odds)
        psa = _to_float_or_none(r.ps_away_odds)
        if psh and psd and psa:
            ps_records.append((r.external_id, psh, psd, psa))

    ps_upsert_sql = """
        INSERT INTO public.match_odds (match_id, bookmaker, home_odds, draw_odds, away_odds)
        SELECT hm.id, 'PS', v.home_odds, v.draw_odds, v.away_odds
        FROM (VALUES %s) AS v(external_id, home_odds, draw_odds, away_odds)
        JOIN public.historical_matches hm ON hm.external_id = v.external_id
        ON CONFLICT (match_id, bookmaker) DO UPDATE SET
            home_odds = EXCLUDED.home_odds,
            draw_odds = EXCLUDED.draw_odds,
            away_odds = EXCLUDED.away_odds
    """

    t_up = time.time()
    conn = psycopg2.connect(uri, sslmode="require", connect_timeout=20)
    try:
        with conn.cursor() as cur:
            for start in range(0, len(records), batch_size):
                batch = records[start:start + batch_size]
                execute_values(cur, insert_sql, batch, page_size=500)
                conn.commit()
            for start in range(0, len(ps_records), batch_size):
                batch = ps_records[start:start + batch_size]
                execute_values(cur, ps_upsert_sql, batch, page_size=500)
                conn.commit()
    finally:
        conn.close()

    print(f"Upserted {len(records):,} matches + {len(ps_records):,} Pinnacle odds rows into Supabase in {time.time() - t_up:.1f}s.")
    return len(records)


def load_from_supabase(uri: str) -> pd.DataFrame:
    """Query public.historical_matches + Pinnacle closing odds directly from Supabase Postgres."""
    import psycopg2

    sql = """
        SELECT
            hm.league_code,
            hm.league_name,
            hm.season,
            hm.match_date,
            hm.home_team,
            hm.away_team,
            hm.home_goals,
            hm.away_goals,
            hm.ht_home_goals,
            hm.ht_away_goals,
            hm.home_shots,
            hm.away_shots,
            hm.home_shots_on_target AS home_sot,
            hm.away_shots_on_target AS away_sot,
            hm.home_corners,
            hm.away_corners,
            hm.home_fouls,
            hm.away_fouls,
            hm.home_yellow_cards AS home_yellow,
            hm.away_yellow_cards AS away_yellow,
            hm.home_red_cards AS home_red,
            hm.away_red_cards AS away_red,
            hm.avg_home_odds,
            hm.avg_draw_odds,
            hm.avg_away_odds,
            hm.over25_odds,
            hm.under25_odds,
            mo.home_odds AS ps_home_odds,
            mo.draw_odds AS ps_draw_odds,
            mo.away_odds AS ps_away_odds
        FROM public.historical_matches hm
        LEFT JOIN public.match_odds mo
            ON mo.match_id = hm.id AND mo.bookmaker = 'PS'
        WHERE hm.home_goals IS NOT NULL
          AND hm.away_goals IS NOT NULL
          AND hm.match_date >= %s
        ORDER BY hm.match_date ASC, hm.league_code ASC, hm.home_team ASC
    """
    conn = psycopg2.connect(uri, sslmode="require", connect_timeout=20)
    try:
        df = pd.read_sql(sql, conn, params=(MIN_MATCH_DATE,))
    finally:
        conn.close()

    if df.empty:
        return df

    df["match_date"] = pd.to_datetime(df["match_date"])
    df["home_goals"] = df["home_goals"].astype(int)
    df["away_goals"] = df["away_goals"].astype(int)
    numeric_cols = [
        "ht_home_goals", "ht_away_goals", "home_shots", "away_shots",
        "home_sot", "away_sot", "home_corners", "away_corners",
        "home_fouls", "away_fouls", "home_yellow", "away_yellow",
        "home_red", "away_red", "avg_home_odds", "avg_draw_odds",
        "avg_away_odds", "over25_odds", "under25_odds",
        "ps_home_odds", "ps_draw_odds", "ps_away_odds",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df["result"] = np.where(
        df["home_goals"] > df["away_goals"], "H",
        np.where(df["home_goals"] < df["away_goals"], "A", "D"),
    )
    print(f"Loaded {len(df):,} matches directly from Supabase Postgres.")
    return df


def _fetch_one_extra_csv(item: tuple[str, str]) -> pd.DataFrame | None:
    league_code, league_name = item
    url = f"https://www.football-data.co.uk/new/{league_code}.csv"
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "PunterStat-Kaggle/1.0"})
        with urllib.request.urlopen(req, timeout=20) as resp:
            raw_bytes = resp.read()
        raw = pd.read_csv(io.BytesIO(raw_bytes), encoding="latin1", on_bad_lines="skip")
        if raw.empty or "Home" not in raw.columns or "Away" not in raw.columns:
            return None
        df = raw.dropna(subset=["Home", "Away", "Date"]).copy()
        match_dates = parse_fdco_date(df["Date"])
        home_teams = df["Home"].astype(str).str.strip()
        away_teams = df["Away"].astype(str).str.strip()
        seasons = df["Season"].astype(str).str.strip() if "Season" in df.columns else match_dates.dt.year.astype(str)

        out = pd.DataFrame({
            "source": "football-data-co-uk-extra",
            "league_code": league_code,
            "league_name": league_name,
            "season": seasons,
            "match_date": match_dates,
            "match_time": df["Time"].astype(str).str.strip() if "Time" in df.columns else None,
            "home_team": home_teams,
            "away_team": away_teams,
            "home_goals": first_col(df, ["HG", "FTHG"]),
            "away_goals": first_col(df, ["AG", "FTAG"]),
            "ht_home_goals": np.nan, "ht_away_goals": np.nan,
            "home_shots": np.nan, "away_shots": np.nan,
            "home_sot": np.nan, "away_sot": np.nan,
            "home_corners": np.nan, "away_corners": np.nan,
            "home_fouls": np.nan, "away_fouls": np.nan,
            "home_yellow": np.nan, "away_yellow": np.nan,
            "home_red": np.nan, "away_red": np.nan,
            "avg_home_odds": first_col(df, ["AvgH", "AvgCH", "B365H"]),
            "avg_draw_odds": first_col(df, ["AvgD", "AvgCD", "B365D"]),
            "avg_away_odds": first_col(df, ["AvgA", "AvgCA", "B365A"]),
            "max_home_odds": first_col(df, ["MaxH", "MaxCH"]),
            "max_draw_odds": first_col(df, ["MaxD", "MaxCD"]),
            "max_away_odds": first_col(df, ["MaxA", "MaxCA"]),
            "over25_odds": np.nan, "under25_odds": np.nan,
            "max_over25_odds": np.nan, "max_under25_odds": np.nan,
            "handi_size": np.nan, "handi_home_odds": np.nan, "handi_away_odds": np.nan,
            "ps_home_odds": first_col(df, ["PSCH", "PH"]),
            "ps_draw_odds": first_col(df, ["PSCD", "PD"]),
            "ps_away_odds": first_col(df, ["PSCA", "PA"]),
        })
        out = out.dropna(subset=["match_date", "home_goals", "away_goals"]).copy()
        out = out[out["match_date"] >= pd.Timestamp(MIN_MATCH_DATE)].copy()
        if out.empty:
            return None
        out["home_goals"] = out["home_goals"].astype(int)
        out["away_goals"] = out["away_goals"].astype(int)
        out["result"] = np.where(
            out["home_goals"] > out["away_goals"], "H",
            np.where(out["home_goals"] < out["away_goals"], "A", "D"),
        )
        out["external_id"] = [
            f"{lc}_{dt.strftime('%Y-%m-%d')}_{slugify(ht)}_{slugify(at)}"
            for lc, dt, ht, at in zip(out["league_code"], out["match_date"], out["home_team"], out["away_team"])
        ]
        return out
    except Exception:
        return None


def download_fdco_extra(extra_leagues: dict[str, str]) -> pd.DataFrame:
    print(f"Downloading {len(extra_leagues)} global Extra leagues from football-data.co.uk/new/...")
    t_ex = time.time()
    frames = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as ex:
        for res in ex.map(_fetch_one_extra_csv, list(extra_leagues.items())):
            if res is not None:
                frames.append(res)
    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["external_id"], keep="last")
    print(f"Downloaded {len(frames)}/{len(extra_leagues)} Extra leagues -> {len(df):,} matches in {time.time() - t_ex:.1f}s.")
    return df


def seed_international_to_supabase(uri: str | None) -> pd.DataFrame:
    """Load Kaggle martj42 international dataset (AFCON, World Cup, Algeria/Egypt/Senegal/Spain) and upsert to Supabase."""
    candidates = [
        Path("/kaggle/input/international-football-results-from-1872-to-2017/results.csv"),
        Path("/kaggle/input/datasets/martj42/international-football-results-from-1872-to-2017/results.csv"),
    ]
    csv_path = next((p for p in candidates if p.exists()), None)
    if csv_path is None:
        print("International Kaggle dataset not mounted — skipping.")
        return pd.DataFrame()

    intl = pd.read_csv(csv_path)
    intl = intl.dropna(subset=["date", "home_team", "away_team", "home_score", "away_score"]).copy()
    intl["home_score"] = pd.to_numeric(intl["home_score"], errors="coerce")
    intl["away_score"] = pd.to_numeric(intl["away_score"], errors="coerce")
    intl = intl.dropna(subset=["home_score", "away_score"]).copy()
    intl["home_score"] = intl["home_score"].astype(int)
    intl["away_score"] = intl["away_score"].astype(int)

    focus = ["Algeria", "Egypt", "Senegal", "Spain"]
    counts = {
        t: int(((intl["home_team"] == t) | (intl["away_team"] == t)).sum())
        for t in focus
    }
    print(f"International dataset loaded: {len(intl):,} matches | Focus national teams: {counts}")

    if not uri:
        return intl

    try:
        import psycopg2
        from psycopg2.extras import execute_values

        sql = """
            INSERT INTO public.international_matches
                (source, external_id, match_date, home_team, away_team,
                 home_score, away_score, result, tournament, city, country, neutral)
            VALUES %s
            ON CONFLICT (external_id) DO UPDATE SET
                home_score = EXCLUDED.home_score,
                away_score = EXCLUDED.away_score,
                result     = EXCLUDED.result,
                tournament = EXCLUDED.tournament
        """
        records = []
        seen_ids = set()
        for r in intl.itertuples(index=False):
            dt_str = str(r.date).strip()
            ext_id = f"{dt_str}_{slugify(r.home_team)}_{slugify(r.away_team)}"
            if ext_id in seen_ids:
                continue
            seen_ids.add(ext_id)
            res = "H" if r.home_score > r.away_score else ("A" if r.home_score < r.away_score else "D")
            neutral = str(getattr(r, "neutral", False)).strip().lower() == "true"
            records.append((
                "kaggle-martj42", ext_id, dt_str, str(r.home_team).strip(), str(r.away_team).strip(),
                int(r.home_score), int(r.away_score), res,
                str(getattr(r, "tournament", "") or "") or None,
                str(getattr(r, "city", "") or "") or None,
                str(getattr(r, "country", "") or "") or None,
                neutral,
            ))

        conn = psycopg2.connect(uri, sslmode="require", connect_timeout=20)
        try:
            with conn.cursor() as cur:
                for start in range(0, len(records), 2000):
                    execute_values(cur, sql, records[start:start + 2000], page_size=500)
                    conn.commit()
        finally:
            conn.close()
        print(f"Upserted {len(records):,} international matches into Supabase public.international_matches.")
    except Exception as exc:
        print(f"Skipping Supabase international_matches upsert ({exc}).")
    return intl


def fetch_sportsapipro_leagues(
    api_key: str,
    target_leagues: dict[str, tuple[str, str, int | None]],
    max_requests: int = SPORTSAPIPRO_MAX_REQUESTS,
) -> pd.DataFrame:
    """Fetch finished matches for up to 18 African, Continental & Global leagues via SportsAPIPro v2.

    Uses canonical uniqueTournament IDs when known (skipping /search), enforces a strict
    request budget (`max_requests`), and stops gracefully on HTTP 429 so already-fetched
    leagues are still upserted to Supabase.
    """
    base = "https://api.sportsapipro.com/v2/football"
    headers = {"x-api-key": api_key, "accept": "application/json", "User-Agent": "PunterStat-Kaggle/2.1"}
    req_count = 0

    def api_get(path: str) -> dict | None:
        nonlocal req_count
        if req_count >= max_requests:
            return None
        req_count += 1
        req = urllib.request.Request(f"{base}{path}", headers=headers)
        with urllib.request.urlopen(req, timeout=15) as resp:
            return json.loads(resp.read().decode("utf-8"))

    rows = []
    fetched_leagues = []
    for league_code, (league_name, query, canonical_tid) in target_leagues.items():
        if req_count >= max_requests:
            print(f"  SportsAPIPro budget reached ({req_count}/{max_requests} reqs) — remaining leagues will seed on next run.")
            break
        try:
            tid = canonical_tid
            if tid is None:
                s_res = api_get(f"/search?q={urllib.parse.quote(query)}")
                if s_res is None:
                    break
                results = s_res.get("results") or s_res.get("data", {}).get("results") or []
                for item in results:
                    entity = item.get("entity") or item
                    if item.get("type") in ("uniqueTournament", "tournament") or "uniqueTournament" in str(item):
                        tid = entity.get("id")
                        if tid:
                            break
            if not tid:
                print(f"  SportsAPIPro: could not resolve tournament id for {league_code} ({league_name})")
                continue

            seasons_res = api_get(f"/tournaments/{tid}/seasons")
            if seasons_res is None:
                break
            seasons = (seasons_res.get("seasons") or [])[:2]  # last 2 seasons per league to maximize league coverage under 100 req/day
            league_rows = 0
            for s in seasons:
                sid = s.get("id")
                s_label = str(s.get("year") or s.get("name") or "")
                ev_res = api_get(f"/tournament/{tid}/season/{sid}/events/last/0")
                if ev_res is None:
                    break
                events = ev_res.get("data", {}).get("events") or []
                for ev in events:
                    if (ev.get("status") or {}).get("type") != "finished":
                        continue
                    hs = (ev.get("homeScore") or {}).get("current")
                    as_ = (ev.get("awayScore") or {}).get("current")
                    ts = ev.get("startTimestamp")
                    ht = (ev.get("homeTeam") or {}).get("name")
                    at = (ev.get("awayTeam") or {}).get("name")
                    if hs is None or as_ is None or not ts or not ht or not at:
                        continue
                    dt = pd.to_datetime(ts, unit="s", utc=True).tz_localize(None)
                    rows.append({
                        "source": "sportsapipro",
                        "external_id": f"{league_code}_{dt.strftime('%Y-%m-%d')}_{slugify(ht)}_{slugify(at)}",
                        "league_code": league_code,
                        "league_name": league_name,
                        "season": s_label or f"{dt.year}/{str(dt.year + 1)[2:]}",
                        "match_date": dt.normalize(),
                        "match_time": dt.strftime("%H:%M"),
                        "home_team": str(ht).strip(),
                        "away_team": str(at).strip(),
                        "home_goals": int(hs),
                        "away_goals": int(as_),
                        "result": "H" if hs > as_ else ("A" if hs < as_ else "D"),
                        "ht_home_goals": np.nan, "ht_away_goals": np.nan,
                        "home_shots": np.nan, "away_shots": np.nan,
                        "home_sot": np.nan, "away_sot": np.nan,
                        "home_corners": np.nan, "away_corners": np.nan,
                        "home_fouls": np.nan, "away_fouls": np.nan,
                        "home_yellow": np.nan, "away_yellow": np.nan,
                        "home_red": np.nan, "away_red": np.nan,
                        "avg_home_odds": np.nan, "avg_draw_odds": np.nan, "avg_away_odds": np.nan,
                        "max_home_odds": np.nan, "max_draw_odds": np.nan, "max_away_odds": np.nan,
                        "over25_odds": np.nan, "under25_odds": np.nan,
                        "max_over25_odds": np.nan, "max_under25_odds": np.nan,
                        "handi_size": np.nan, "handi_home_odds": np.nan, "handi_away_odds": np.nan,
                        "ps_home_odds": np.nan, "ps_draw_odds": np.nan, "ps_away_odds": np.nan,
                    })
                    league_rows += 1
            if league_rows > 0:
                fetched_leagues.append(f"{league_code}({league_rows})")
        except urllib.error.HTTPError as exc:
            print(f"  SportsAPIPro HTTP {exc.code} for {league_code} ({league_name})")
            if exc.code == 429:
                print("  SportsAPIPro rate-limit (429) hit — keeping all matches fetched so far.")
                break
        except Exception as exc:
            print(f"  SportsAPIPro error for {league_code} ({league_name}): {exc}")

    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows).drop_duplicates(subset=["external_id"], keep="last").reset_index(drop=True)
    print(f"Fetched {len(df):,} SportsAPIPro matches across {len(fetched_leagues)} leagues ({req_count} reqs): {', '.join(fetched_leagues)}")
    return df


# ── Main data pipeline (Supabase-first: 22 FDCO Europe + 16 FDCO Extra + 18 SportsAPIPro + International) ──
raw_sb_uri = get_secret("SUPABASE_DB_URL")
sportsapipro_key = get_secret("SPORTSAPIPRO_API_KEY")
sb_uri = sanitize_pg_uri(raw_sb_uri) if raw_sb_uri else None

# 1. Seed international dataset (AFCON, World Cup, Euros, Copa America)
if ENABLE_INTERNATIONAL_SEED:
    intl_df = seed_international_to_supabase(sb_uri)

# 2. Load / auto-seed domestic & continental leagues (up to 56 leagues total)
if CACHE_PARQUET.exists() and not FORCE_REFRESH_SUPABASE:
    print(f"Loading cached dataset from {CACHE_PARQUET}")
    matches = pd.read_parquet(CACHE_PARQUET)
elif sb_uri:
    matches = load_from_supabase(sb_uri)
    existing_leagues = set(matches["league_code"].unique()) if not matches.empty else set()
    missing_core = {k: v for k, v in LEAGUES.items() if k not in existing_leagues}
    missing_extra = {k: v for k, v in EXTRA_LEAGUES.items() if k not in existing_leagues} if ENABLE_EXTRA_LEAGUES else {}
    missing_sap = {k: v for k, v in SPORTSAPIPRO_LEAGUES.items() if k not in existing_leagues} if ENABLE_SPORTSAPIPRO_LEAGUES else {}

    to_upsert = []
    if FORCE_REFRESH_SUPABASE or (matches.empty and SEED_SUPABASE_IF_EMPTY):
        to_upsert.append(download_fdco_all(LEAGUES, SEED_SEASONS))
        if ENABLE_EXTRA_LEAGUES:
            to_upsert.append(download_fdco_extra(EXTRA_LEAGUES))
    else:
        if missing_core and SEED_SUPABASE_IF_EMPTY:
            print(f"Auto-seeding {len(missing_core)} missing FDCO Europe leagues: {sorted(missing_core)}")
            to_upsert.append(download_fdco_all(missing_core, SEED_SEASONS))
        if missing_extra and SEED_SUPABASE_IF_EMPTY:
            print(f"Auto-seeding {len(missing_extra)} missing FDCO Extra leagues: {sorted(missing_extra)}")
            to_upsert.append(download_fdco_extra(missing_extra))

    if ENABLE_SPORTSAPIPRO_LEAGUES and sportsapipro_key:
        sap_targets = SPORTSAPIPRO_LEAGUES if FORCE_REFRESH_SUPABASE else missing_sap
        if sap_targets:
            print(f"Auto-seeding {len(sap_targets)} SportsAPIPro leagues: {sorted(sap_targets)}")
            sap_df = fetch_sportsapipro_leagues(sportsapipro_key, sap_targets)
            if not sap_df.empty:
                to_upsert.append(sap_df)
    elif ENABLE_SPORTSAPIPRO_LEAGUES:
        print("SPORTSAPIPRO_API_KEY not attached — skipping African/Continental/Global SportsAPIPro leagues (attach secret to enable).")

    if to_upsert:
        combined = pd.concat([d for d in to_upsert if not d.empty], ignore_index=True)
        if not combined.empty:
            upsert_to_supabase(combined, sb_uri)
            matches = load_from_supabase(sb_uri)

    matches = (
        matches
        .drop_duplicates(subset=["league_code", "match_date", "home_team", "away_team"])
        .sort_values(["match_date", "league_code", "home_team"])
        .reset_index(drop=True)
    )
    matches.to_parquet(CACHE_PARQUET, index=False)
else:
    print("SUPABASE_DB_URL secret not attached — running in-memory from football-data.co.uk CSVs.")
    parts = [download_fdco_all(LEAGUES, SEED_SEASONS)]
    if ENABLE_EXTRA_LEAGUES:
        parts.append(download_fdco_extra(EXTRA_LEAGUES))
    if ENABLE_SPORTSAPIPRO_LEAGUES and sportsapipro_key:
        sap_df = fetch_sportsapipro_leagues(sportsapipro_key, SPORTSAPIPRO_LEAGUES)
        if not sap_df.empty:
            parts.append(sap_df)
    matches = (
        pd.concat([p for p in parts if not p.empty], ignore_index=True)
        .drop_duplicates(subset=["league_code", "match_date", "home_team", "away_team"])
        .sort_values(["match_date", "league_code", "home_team"])
        .reset_index(drop=True)
    )
    matches.to_parquet(CACHE_PARQUET, index=False)

matches = matches[matches["match_date"] >= pd.Timestamp(MIN_MATCH_DATE)].reset_index(drop=True)
print(f"Total matches: {len(matches):,} | Date range: {matches['match_date'].min().date()} → {matches['match_date'].max().date()}")
print(f"Leagues ({matches['league_code'].nunique()}): {sorted(matches['league_code'].unique())}")
matches.tail(5)

## 2. Leakage-Free Feature Engineering (Elo + Rolling Pre-Match Form)

Every feature is computed strictly from **past matches** (`shift(1)` / sequential Elo update before updating with the current match's score) so there is zero target leakage.

In [ ]:
def compute_elo_ratings(
    df: pd.DataFrame,
    base_elo: float = 1500.0,
    k_factor: float = 20.0,
    home_adv: float = 65.0,
    season_regression: float = 0.20,
) -> pd.DataFrame:
    """Compute pre-match Elo ratings chronologically with mild between-season mean reversion."""
    ratings: dict[tuple[str, str], float] = {}
    last_season: dict[tuple[str, str], str] = {}

    home_elos = np.empty(len(df), dtype=np.float64)
    away_elos = np.empty(len(df), dtype=np.float64)

    for i, row in enumerate(df.itertuples(index=False)):
        hk = (row.league_code, row.home_team)
        ak = (row.league_code, row.away_team)

        h_elo = ratings.get(hk, base_elo)
        a_elo = ratings.get(ak, base_elo)

        # Mild mean reversion at the start of a new season
        if hk in last_season and last_season[hk] != row.season:
            h_elo = h_elo * (1.0 - season_regression) + base_elo * season_regression
        if ak in last_season and last_season[ak] != row.season:
            a_elo = a_elo * (1.0 - season_regression) + base_elo * season_regression

        home_elos[i] = h_elo
        away_elos[i] = a_elo

        # Expected score (home advantage included)
        exp_home = 1.0 / (1.0 + 10.0 ** ((a_elo - (h_elo + home_adv)) / 400.0))
        if row.home_goals > row.away_goals:
            actual_home = 1.0
        elif row.home_goals < row.away_goals:
            actual_home = 0.0
        else:
            actual_home = 0.5

        # Goal-difference multiplier (rewards decisive wins slightly more)
        gd = abs(row.home_goals - row.away_goals)
        mov_mult = math.log(gd + 1.0) * (2.2 / ((h_elo - a_elo) * 0.001 + 2.2)) if gd > 0 else 1.0
        delta = k_factor * max(1.0, mov_mult) * (actual_home - exp_home)

        ratings[hk] = h_elo + delta
        ratings[ak] = a_elo - delta
        last_season[hk] = row.season
        last_season[ak] = row.season

    return pd.DataFrame({
        "home_elo": home_elos,
        "away_elo": away_elos,
        "elo_diff": home_elos - away_elos,
        "elo_home_exp": 1.0 / (1.0 + 10.0 ** ((away_elos - (home_elos + home_adv)) / 400.0)),
    }, index=df.index)


def build_rolling_team_features(
    df: pd.DataFrame,
    elo_df: pd.DataFrame,
    windows: tuple[int, ...] = (5, 10, 20),
) -> pd.DataFrame:
    """v2.0 Leakage-free rolling features: attacking, HT/2H, defensive, efficiency, consistency (std), and Elo-adjusted form."""
    ht_h = pd.to_numeric(df["ht_home_goals"], errors="coerce")
    ht_a = pd.to_numeric(df["ht_away_goals"], errors="coerce")
    sh_h = np.where(ht_h.notna(), df["home_goals"] - ht_h, np.nan)
    sh_a = np.where(ht_a.notna(), df["away_goals"] - ht_a, np.nan)

    h_pts = np.where(df["home_goals"] > df["away_goals"], 3.0, np.where(df["home_goals"] == df["away_goals"], 1.0, 0.0))
    a_pts = np.where(df["away_goals"] > df["home_goals"], 3.0, np.where(df["away_goals"] == df["home_goals"], 1.0, 0.0))

    h_opp_mult = np.clip(1.0 + (elo_df["away_elo"].values - 1500.0) / 600.0, 0.5, 1.5)
    a_opp_mult = np.clip(1.0 + (elo_df["home_elo"].values - 1500.0) / 600.0, 0.5, 1.5)

    h_sot = pd.to_numeric(df["home_sot"], errors="coerce")
    a_sot = pd.to_numeric(df["away_sot"], errors="coerce")
    h_sh  = pd.to_numeric(df["home_shots"], errors="coerce")
    a_sh  = pd.to_numeric(df["away_shots"], errors="coerce")

    home_view = pd.DataFrame({
        "match_idx": df.index,
        "match_date": df["match_date"],
        "league_code": df["league_code"],
        "team": df["home_team"],
        "is_home": 1,
        "gf": df["home_goals"],
        "ga": df["away_goals"],
        "ht_gf": ht_h,
        "ht_ga": ht_a,
        "sh_gf": sh_h,
        "sh_ga": sh_a,
        "clean_sheet": (df["away_goals"] == 0).astype(float),
        "failed_to_score": (df["home_goals"] == 0).astype(float),
        "shots_for": h_sh,
        "shots_ag": a_sh,
        "sot_for": h_sot,
        "sot_ag": a_sot,
        "shot_conv": np.where(h_sot > 0, df["home_goals"] / h_sot, np.nan),
        "shot_acc": np.where(h_sh > 0, h_sot / h_sh, np.nan),
        "corners_for": df["home_corners"],
        "corners_ag": df["away_corners"],
        "fouls": df["home_fouls"],
        "yellow": df["home_yellow"],
        "red": df["home_red"],
        "points": h_pts,
        "opp_elo": elo_df["away_elo"].values,
        "elo_adj_pts": h_pts * h_opp_mult,
    })

    away_view = pd.DataFrame({
        "match_idx": df.index,
        "match_date": df["match_date"],
        "league_code": df["league_code"],
        "team": df["away_team"],
        "is_home": 0,
        "gf": df["away_goals"],
        "ga": df["home_goals"],
        "ht_gf": ht_a,
        "ht_ga": ht_h,
        "sh_gf": sh_a,
        "sh_ga": sh_h,
        "clean_sheet": (df["home_goals"] == 0).astype(float),
        "failed_to_score": (df["away_goals"] == 0).astype(float),
        "shots_for": a_sh,
        "shots_ag": h_sh,
        "sot_for": a_sot,
        "sot_ag": h_sot,
        "shot_conv": np.where(a_sot > 0, df["away_goals"] / a_sot, np.nan),
        "shot_acc": np.where(a_sh > 0, a_sot / a_sh, np.nan),
        "corners_for": df["away_corners"],
        "corners_ag": df["home_corners"],
        "fouls": df["away_fouls"],
        "yellow": df["away_yellow"],
        "red": df["away_red"],
        "points": a_pts,
        "opp_elo": elo_df["home_elo"].values,
        "elo_adj_pts": a_pts * a_opp_mult,
    })

    long_df = pd.concat([home_view, away_view], ignore_index=True)
    long_df = long_df.sort_values(["team", "match_date", "match_idx"]).reset_index(drop=True)

    grp = long_df.groupby("team", sort=False)

    # Rest days since previous match (clipped at 30 days)
    long_df["rest_days"] = grp["match_date"].diff().dt.days.clip(lower=1, upper=30)
    long_df["matches_played"] = grp.cumcount()

    # Core rolling means across [5, 10, 20]
    core_cols = ["gf", "ga", "shots_for", "shots_ag", "sot_for", "sot_ag", "corners_for", "points"]
    feat_cols = ["rest_days", "matches_played"]

    for w in windows:
        for col in core_cols:
            fname = f"{col}_roll{w}"
            long_df[fname] = grp[col].transform(lambda s: s.shift(1).rolling(w, min_periods=2).mean())
            feat_cols.append(fname)

    # Extended v2.0 rolling means (window=10: HT/2H, efficiency, clean sheets, cards, opponent strength)
    ext_cols = [
        "ht_gf", "ht_ga", "sh_gf", "sh_ga",
        "clean_sheet", "failed_to_score",
        "shot_conv", "shot_acc", "corners_ag",
        "fouls", "yellow", "red", "opp_elo", "elo_adj_pts",
    ]
    for col in ext_cols:
        fname = f"{col}_roll10"
        long_df[fname] = grp[col].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean())
        feat_cols.append(fname)

    # Team consistency / volatility (rolling standard deviation over last 10 games)
    for col in ["gf", "ga", "points"]:
        fname = f"{col}_std10"
        long_df[fname] = grp[col].transform(lambda s: s.shift(1).rolling(10, min_periods=4).std())
        feat_cols.append(fname)

    # Venue-specific rolling goals & points (home-only for home team, away-only for away team)
    vgrp = long_df.groupby(["team", "is_home"], sort=False)
    for col in ["gf", "ga", "points", "clean_sheet"]:
        fname = f"{col}_venue_roll8"
        long_df[fname] = vgrp[col].transform(lambda s: s.shift(1).rolling(8, min_periods=2).mean())
        feat_cols.append(fname)

    # Split back to home and away rows indexed by match_idx
    home_feats = (
        long_df[long_df["is_home"] == 1]
        .set_index("match_idx")[feat_cols]
        .add_prefix("h_")
    )
    away_feats = (
        long_df[long_df["is_home"] == 0]
        .set_index("match_idx")[feat_cols]
        .add_prefix("a_")
    )

    out = pd.concat([home_feats, away_feats], axis=1).reindex(df.index)

    # High-signal difference features
    for w in windows:
        out[f"diff_gf_roll{w}"] = out[f"h_gf_roll{w}"] - out[f"a_gf_roll{w}"]
        out[f"diff_ga_roll{w}"] = out[f"h_ga_roll{w}"] - out[f"a_ga_roll{w}"]
        out[f"diff_pts_roll{w}"] = out[f"h_points_roll{w}"] - out[f"a_points_roll{w}"]
        out[f"diff_sot_roll{w}"] = out[f"h_sot_for_roll{w}"] - out[f"a_sot_for_roll{w}"]
    out["diff_elo_adj_pts10"] = out["h_elo_adj_pts_roll10"] - out["a_elo_adj_pts_roll10"]
    out["diff_clean_sheet10"] = out["h_clean_sheet_roll10"] - out["a_clean_sheet_roll10"]
    out["diff_gf_std10"]      = out["h_gf_std10"] - out["a_gf_std10"]
    out["diff_rest_days"]     = out["h_rest_days"] - out["a_rest_days"]

    return out


def add_market_probs(df: pd.DataFrame) -> pd.DataFrame:
    """De-vig closing 1X2, Pinnacle, and Over/Under 2.5 odds into fair implied probabilities."""
    out = pd.DataFrame(index=df.index)
    inv_h = 1.0 / df["avg_home_odds"]
    inv_d = 1.0 / df["avg_draw_odds"]
    inv_a = 1.0 / df["avg_away_odds"]
    overround = inv_h + inv_d + inv_a
    valid_1x2 = (overround > 0.95) & (overround < 1.30)

    out["mkt_p_home"] = np.where(valid_1x2, inv_h / overround, np.nan)
    out["mkt_p_draw"] = np.where(valid_1x2, inv_d / overround, np.nan)
    out["mkt_p_away"] = np.where(valid_1x2, inv_a / overround, np.nan)
    out["mkt_overround"] = np.where(valid_1x2, overround - 1.0, np.nan)

    # Pinnacle sharp closing line (PS-05 benchmark)
    if {"ps_home_odds", "ps_draw_odds", "ps_away_odds"}.issubset(df.columns):
        ps_ih = 1.0 / df["ps_home_odds"]
        ps_id = 1.0 / df["ps_draw_odds"]
        ps_ia = 1.0 / df["ps_away_odds"]
        ps_sum = ps_ih + ps_id + ps_ia
        valid_ps = (ps_sum > 0.95) & (ps_sum < 1.20)
        out["ps_p_home"] = np.where(valid_ps, ps_ih / ps_sum, np.nan)
        out["ps_p_draw"] = np.where(valid_ps, ps_id / ps_sum, np.nan)
        out["ps_p_away"] = np.where(valid_ps, ps_ia / ps_sum, np.nan)
    else:
        out["ps_p_home"] = np.nan
        out["ps_p_draw"] = np.nan
        out["ps_p_away"] = np.nan

    inv_o = 1.0 / df["over25_odds"]
    inv_u = 1.0 / df["under25_odds"]
    ou_sum = inv_o + inv_u
    valid_ou = (ou_sum > 0.95) & (ou_sum < 1.30)
    out["mkt_p_over25"] = np.where(valid_ou, inv_o / ou_sum, np.nan)
    return out


# ── Data Quality & Per-League Reliability Report (v2.0) ───────────────────────
invalid_scores = int(((matches["home_goals"] < 0) | (matches["away_goals"] < 0) | (matches["home_goals"] > 20)).sum())
dup_count = int(matches.duplicated(subset=["league_code", "match_date", "home_team", "away_team"]).sum())
hash_payload = f"{len(matches)}|{matches['match_date'].min()}|{matches['match_date'].max()}|{matches['home_goals'].sum()}|{matches['away_goals'].sum()}"
DATASET_SHA256 = hashlib.sha256(hash_payload.encode("utf-8")).hexdigest()[:16]

reliability_df = matches.groupby(["league_code", "league_name"]).agg(
    matches=("match_date", "count"),
    seasons=("season", "nunique"),
    odds_pct=("avg_home_odds", lambda s: round(100.0 * s.notna().mean(), 1)),
    pinnacle_pct=("ps_home_odds", lambda s: round(100.0 * s.notna().mean(), 1)) if "ps_home_odds" in matches.columns else ("avg_home_odds", lambda s: 0.0),
    shots_pct=("home_shots", lambda s: round(100.0 * s.notna().mean(), 1)),
    ht_pct=("ht_home_goals", lambda s: round(100.0 * s.notna().mean(), 1)),
).reset_index()

reliability_df["reliability_tier"] = np.where(
    (reliability_df["matches"] >= 4000) & (reliability_df["odds_pct"] >= 95.0) & (reliability_df["shots_pct"] >= 80.0),
    "Gold",
    np.where((reliability_df["matches"] >= 2000) & (reliability_df["odds_pct"] >= 80.0), "Silver", "Bronze"),
)
LEAGUE_RELIABILITY = dict(zip(reliability_df["league_code"], reliability_df["reliability_tier"]))
print(f"Data Quality Audit | SHA256: {DATASET_SHA256} | Invalid scores: {invalid_scores} | Duplicates: {dup_count}")
print("League reliability tiers:", reliability_df["reliability_tier"].value_counts().to_dict())

t0 = time.time()
elo_df = compute_elo_ratings(matches)
roll_df = build_rolling_team_features(matches, elo_df)
mkt_df = add_market_probs(matches)

data = pd.concat([matches, elo_df, roll_df, mkt_df], axis=1)
data["league_id"] = data["league_code"].astype("category").cat.codes

# Target encoding: 0 = home_win, 1 = draw, 2 = away_win (matches PunterStat scorer order)
TARGET_MAP = {"H": 0, "D": 1, "A": 2}
data["target_1x2"] = data["result"].map(TARGET_MAP).astype(int)
data["target_over15"] = ((data["home_goals"] + data["away_goals"]) >= 2).astype(int)
data["target_over25"] = ((data["home_goals"] + data["away_goals"]) >= 3).astype(int)
data["target_over35"] = ((data["home_goals"] + data["away_goals"]) >= 4).astype(int)
data["target_btts"]   = ((data["home_goals"] > 0) & (data["away_goals"] > 0)).astype(int)

# Drop very early matches where teams have < 3 games of history
data = data[(data["h_matches_played"] >= 3) & (data["a_matches_played"] >= 3)].reset_index(drop=True)

STATS_FEATURES = (
    ["league_id", "home_elo", "away_elo", "elo_diff", "elo_home_exp"]
    + list(roll_df.columns)
)
MARKET_FEATURES = ["mkt_p_home", "mkt_p_draw", "mkt_p_away", "mkt_p_over25"]
BLENDED_FEATURES = STATS_FEATURES + MARKET_FEATURES

# Nested chronological splits: train -> tune (early stopping & xi) -> cal (temperature, rho, alpha) -> test
train_df = data[data["match_date"] < TRAIN_END_DATE].copy()
tune_df  = data[(data["match_date"] >= TRAIN_END_DATE) & (data["match_date"] < TUNE_END_DATE)].copy()
cal_df   = data[(data["match_date"] >= TUNE_END_DATE) & (data["match_date"] < CAL_END_DATE)].copy()
val_df   = cal_df  # calibration split (strictly separate from tune_df!)
test_df  = data[data["match_date"] >= CAL_END_DATE].copy()

print(f"v2.0 Feature engineering completed in {time.time() - t0:.1f}s")
print(f"Pure-stats features: {len(STATS_FEATURES)} | Blended features: {len(BLENDED_FEATURES)}")
print(
    f"Nested Splits -> Train: {len(train_df):,} | Tune (22/23): {len(tune_df):,} | "
    f"Cal (23/24): {len(cal_df):,} | Test (24/25+): {len(test_df):,}"
)

## 3. PunterStat Evaluation Metrics (`lib/calibration/scorer.ts` Parity)

- **Multi-class Brier Score** (range $0 \to 2$, lower is better):
  $$\text{Brier} = \frac{1}{N} \sum_{i=1}^{N} \sum_{k \in \{H, D, A\}} (p_{i,k} - y_{i,k})^2$$
- **Ranked Probability Score (RPS)**: respects the natural ordering $H \to D \to A$ (lower is better).
- **Multi-class Log-Loss**: strictly proper scoring rule penalizing overconfident misses.
- **5-Bucket Reliability Curve**: exact `0-20%`, `20-40%`, `40-60%`, `60-80%`, `80-100%` bins from `scorer.ts`.

In [ ]:
def multiclass_brier_score(y_true: np.ndarray, probs: np.ndarray) -> float:
    """Exact match to artifacts/punterstat/lib/calibration/scorer.ts brierScore()."""
    y_oh = np.eye(3, dtype=np.float64)[y_true]
    return float(np.mean(np.sum((probs - y_oh) ** 2, axis=1)))


def multiclass_log_loss(y_true: np.ndarray, probs: np.ndarray, eps: float = 1e-15) -> float:
    p = np.clip(probs, eps, 1.0 - eps)
    p = p / p.sum(axis=1, keepdims=True)
    return float(-np.mean(np.log(p[np.arange(len(y_true)), y_true])))


def ranked_probability_score(y_true: np.ndarray, probs: np.ndarray) -> float:
    """Ordinal RPS for 3 outcomes [H, D, A]. Lower is better."""
    y_oh = np.eye(3, dtype=np.float64)[y_true]
    cum_p = np.cumsum(probs, axis=1)[:, :2]
    cum_y = np.cumsum(y_oh, axis=1)[:, :2]
    return float(np.mean(0.5 * np.sum((cum_p - cum_y) ** 2, axis=1)))


def binary_brier_score(y_true: np.ndarray, probs: np.ndarray) -> float:
    return float(np.mean((probs - y_true) ** 2))


def calibration_curve_punterstat(y_true: np.ndarray, probs: np.ndarray) -> pd.DataFrame:
    """Exact 5-bucket reliability curve matching scorer.ts."""
    y_oh = np.eye(3, dtype=np.float64)[y_true]
    flat_p = probs.ravel()
    flat_y = y_oh.ravel()

    buckets = [
        ("0-20%", 0.0, 0.2),
        ("20-40%", 0.2, 0.4),
        ("40-60%", 0.4, 0.6),
        ("60-80%", 0.6, 0.8),
        ("80-100%", 0.8, 1.01),
    ]
    rows = []
    for label, lo, hi in buckets:
        mask = (flat_p >= lo) & (flat_p < hi)
        cnt = int(mask.sum())
        rows.append({
            "bucket": label,
            "predicted_avg": float(flat_p[mask].mean()) if cnt > 0 else np.nan,
            "actual_freq": float(flat_y[mask].mean()) if cnt > 0 else np.nan,
            "sample_size": cnt,
        })
    return pd.DataFrame(rows)


def evaluate_1x2(name: str, y_true: np.ndarray, probs: np.ndarray) -> dict:
    mask = ~np.isnan(probs).any(axis=1)
    y = y_true[mask]
    p = probs[mask]
    p = p / p.sum(axis=1, keepdims=True)
    return {
        "model": name,
        "n_matches": int(len(y)),
        "brier_3class": round(multiclass_brier_score(y, p), 5),
        "log_loss": round(multiclass_log_loss(y, p), 5),
        "rps": round(ranked_probability_score(y, p), 5),
        "accuracy": round(float(np.mean(np.argmax(p, axis=1) == y)), 4),
    }

## 4. Vectorized Dixon-Coles Score Matrix & Per-League $\rho$ Fitting

Matches `artifacts/punterstat/lib/match-breakdown/analyzer.ts` (`MAX_GOALS = 8`, `DC_RHO = -0.135`), and adds a maximum-likelihood fitter so we can test whether fitting $\rho$ per league improves calibration.

In [ ]:
MAX_GOALS = 8
DEFAULT_DC_RHO = -0.135


def dixon_coles_markets(
    lam_h: np.ndarray,
    lam_a: np.ndarray,
    rho: float | np.ndarray = DEFAULT_DC_RHO,
    max_goals: int = MAX_GOALS,
) -> dict[str, np.ndarray]:
    """Compute 1X2, Over 2.5, and BTTS probabilities from Poisson lambdas + Dixon-Coles tau."""
    lam_h = np.clip(np.asarray(lam_h, dtype=np.float64), 0.05, 6.0)
    lam_a = np.clip(np.asarray(lam_a, dtype=np.float64), 0.05, 6.0)
    rho_arr = np.broadcast_to(np.asarray(rho, dtype=np.float64), lam_h.shape)

    goals = np.arange(max_goals + 1)
    ph = poisson.pmf(goals[None, :], lam_h[:, None])  # (N, G)
    pa = poisson.pmf(goals[None, :], lam_a[:, None])  # (N, G)
    mat = ph[:, :, None] * pa[:, None, :]             # (N, G, G)

    # Apply Dixon-Coles low-score tau adjustment
    mat[:, 0, 0] *= np.clip(1.0 - lam_h * lam_a * rho_arr, 1e-6, None)
    mat[:, 0, 1] *= np.clip(1.0 + lam_h * rho_arr, 1e-6, None)
    mat[:, 1, 0] *= np.clip(1.0 + lam_a * rho_arr, 1e-6, None)
    mat[:, 1, 1] *= np.clip(1.0 - rho_arr, 1e-6, None)

    # Re-normalize each match matrix
    totals = mat.sum(axis=(1, 2), keepdims=True)
    mat = np.divide(mat, totals, where=totals > 0)

    i_grid, j_grid = np.meshgrid(goals, goals, indexing="ij")
    tot_grid = i_grid + j_grid
    diff_grid = i_grid - j_grid

    p_home = mat[:, i_grid > j_grid].sum(axis=1)
    p_draw = mat[:, i_grid == j_grid].sum(axis=1)
    p_away = mat[:, i_grid < j_grid].sum(axis=1)
    non_draw = np.clip(p_home + p_away, 1e-9, None)

    return {
        "probs_1x2": np.column_stack([p_home, p_draw, p_away]),
        # Double chance & Draw No Bet
        "p_1x": p_home + p_draw,
        "p_x2": p_away + p_draw,
        "p_12": p_home + p_away,
        "p_home_dnb": p_home / non_draw,
        "p_away_dnb": p_away / non_draw,
        # Goal totals (Over 0.5 .. 4.5)
        "p_over05": mat[:, tot_grid >= 1].sum(axis=1),
        "p_over15": mat[:, tot_grid >= 2].sum(axis=1),
        "p_over25": mat[:, tot_grid >= 3].sum(axis=1),
        "p_over35": mat[:, tot_grid >= 4].sum(axis=1),
        "p_over45": mat[:, tot_grid >= 5].sum(axis=1),
        # BTTS & Clean sheets
        "p_btts": mat[:, (i_grid > 0) & (j_grid > 0)].sum(axis=1),
        "p_home_cs": mat[:, j_grid == 0].sum(axis=1),
        "p_away_cs": mat[:, i_grid == 0].sum(axis=1),
        # Team goal totals
        "p_home_over05": mat[:, i_grid >= 1].sum(axis=1),
        "p_home_over15": mat[:, i_grid >= 2].sum(axis=1),
        "p_away_over05": mat[:, j_grid >= 1].sum(axis=1),
        "p_away_over15": mat[:, j_grid >= 2].sum(axis=1),
        # Asian handicaps (Home -1.5, -0.5, +0.5, +1.5)
        "p_ah_home_m15": mat[:, diff_grid >= 2].sum(axis=1),
        "p_ah_home_m05": p_home,
        "p_ah_home_p05": p_home + p_draw,
        "p_ah_home_p15": mat[:, diff_grid >= -1].sum(axis=1),
    }


def top_scorelines(lam_h: float, lam_a: float, rho: float = DEFAULT_DC_RHO, k: int = 5) -> list[dict]:
    """Return top-k most probable exact scorelines for a single match."""
    goals = np.arange(MAX_GOALS + 1)
    ph = poisson.pmf(goals, max(0.05, lam_h))
    pa = poisson.pmf(goals, max(0.05, lam_a))
    mat = np.outer(ph, pa)
    mat[0, 0] *= max(1e-6, 1.0 - lam_h * lam_a * rho)
    mat[0, 1] *= max(1e-6, 1.0 + lam_h * rho)
    mat[1, 0] *= max(1e-6, 1.0 + lam_a * rho)
    mat[1, 1] *= max(1e-6, 1.0 - rho)
    mat /= mat.sum()
    flat_idx = np.argsort(mat.ravel())[::-1][:k]
    return [
        {"score": f"{idx // (MAX_GOALS + 1)}-{idx % (MAX_GOALS + 1)}", "prob": round(float(mat.ravel()[idx]), 4)}
        for idx in flat_idx
    ]


def fit_dixon_coles_rho(
    lam_h: np.ndarray,
    lam_a: np.ndarray,
    y_hg: np.ndarray,
    y_ag: np.ndarray,
) -> float:
    """Fit Dixon-Coles rho by minimizing negative log-likelihood of observed scorelines."""
    lam_h = np.clip(lam_h, 0.05, 6.0)
    lam_a = np.clip(lam_a, 0.05, 6.0)
    hg = np.clip(y_hg, 0, MAX_GOALS)
    ag = np.clip(y_ag, 0, MAX_GOALS)

    def nll(rho: float) -> float:
        tau = np.ones(len(hg), dtype=np.float64)
        m00 = (hg == 0) & (ag == 0)
        m01 = (hg == 0) & (ag == 1)
        m10 = (hg == 1) & (ag == 0)
        m11 = (hg == 1) & (ag == 1)
        tau[m00] = 1.0 - lam_h[m00] * lam_a[m00] * rho
        tau[m01] = 1.0 + lam_h[m01] * rho
        tau[m10] = 1.0 + lam_a[m10] * rho
        tau[m11] = 1.0 - rho
        if np.any(tau <= 1e-6):
            return 1e9
        return float(-np.mean(np.log(tau)))

    res = minimize_scalar(nll, bounds=(-0.35, 0.15), method="bounded")
    return float(res.x)

## 5. Model 1 — Dual LightGBM Poisson (`lambda_home`, `lambda_away`) + Dixon-Coles

Uses **nested chronological splits**: early stopping runs on `tune_df` (`2022/23`), while per-league Dixon-Coles $\rho$ is fitted strictly on `cal_df` (`2023/24`) so calibration never overfits the tuning set.

In [ ]:
POISSON_PARAMS = {
    "objective": "poisson",
    "metric": "poisson",
    "learning_rate": 0.03,
    "num_leaves": 31,
    "min_child_samples": 60,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_alpha": 1.0,
    "reg_lambda": 5.0,
    "n_estimators": 1500,
    "random_state": 42,
    "verbose": -1,
}

lgb_home = lgb.LGBMRegressor(**POISSON_PARAMS)
lgb_home.fit(
    train_df[STATS_FEATURES], train_df["home_goals"],
    eval_set=[(tune_df[STATS_FEATURES], tune_df["home_goals"])],
    callbacks=[lgb.early_stopping(50, verbose=False)],
)

lgb_away = lgb.LGBMRegressor(**POISSON_PARAMS)
lgb_away.fit(
    train_df[STATS_FEATURES], train_df["away_goals"],
    eval_set=[(tune_df[STATS_FEATURES], tune_df["away_goals"])],
    callbacks=[lgb.early_stopping(50, verbose=False)],
)

print(f"Best iterations -> Home Poisson: {lgb_home.best_iteration_} | Away Poisson: {lgb_away.best_iteration_}")

# Validation predictions to fit global and per-league Dixon-Coles rho
val_lam_h = lgb_home.predict(val_df[STATS_FEATURES])
val_lam_a = lgb_away.predict(val_df[STATS_FEATURES])

global_rho = fit_dixon_coles_rho(val_lam_h, val_lam_a, val_df["home_goals"].values, val_df["away_goals"].values)
print(f"Validation-fitted global DC_RHO: {global_rho:.4f} (current PunterStat constant: {DEFAULT_DC_RHO})")

# Per-league rho (shrunk toward global_rho for leagues with fewer validation matches)
league_rhos = {}
for lc, grp_df in val_df.groupby("league_code"):
    idx = grp_df.index
    pos = val_df.index.get_indexer(idx)
    if len(pos) >= 150:
        raw_rho = fit_dixon_coles_rho(
            val_lam_h[pos], val_lam_a[pos],
            grp_df["home_goals"].values, grp_df["away_goals"].values,
        )
        # Empirical-Bayes shrinkage toward global_rho
        weight = len(pos) / (len(pos) + 200.0)
        league_rhos[lc] = round(weight * raw_rho + (1.0 - weight) * global_rho, 4)
    else:
        league_rhos[lc] = round(global_rho, 4)

print("Per-league fitted DC_RHO:", league_rhos)

# Predict on held-out test set
test_lam_h = lgb_home.predict(test_df[STATS_FEATURES])
test_lam_a = lgb_away.predict(test_df[STATS_FEATURES])

dc_default_test = dixon_coles_markets(test_lam_h, test_lam_a, rho=DEFAULT_DC_RHO)
test_league_rho_vec = test_df["league_code"].map(league_rhos).fillna(global_rho).values
dc_fitted_test = dixon_coles_markets(test_lam_h, test_lam_a, rho=test_league_rho_vec)

## 6. Model 2 — Direct 3-Class LightGBM (`1X2`) + Temperature Scaling

We train two 3-class classifiers:
1. **Pure-Stats 1X2** (no odds) + scalar **Temperature Scaling** $T$ fitted on the validation season.
2. **Stats + Market Blended 1X2** (includes de-vigged bookmaker probabilities) to see whether team/Elo form adds signal on top of the closing market.

In [ ]:
CLF_PARAMS = {
    "objective": "multiclass",
    "num_class": 3,
    "metric": "multi_logloss",
    "learning_rate": 0.03,
    "num_leaves": 31,
    "min_child_samples": 60,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_alpha": 1.0,
    "reg_lambda": 5.0,
    "n_estimators": 1500,
    "random_state": 42,
    "verbose": -1,
}

# 6a. Pure-stats 3-class LightGBM (early stop on tune_df, calibrate on cal_df)
lgb_1x2_stats = lgb.LGBMClassifier(**CLF_PARAMS)
lgb_1x2_stats.fit(
    train_df[STATS_FEATURES], train_df["target_1x2"],
    eval_set=[(tune_df[STATS_FEATURES], tune_df["target_1x2"])],
    callbacks=[lgb.early_stopping(50, verbose=False)],
)


def fit_temperature(raw_logits: np.ndarray, y_true: np.ndarray) -> float:
    """Fit a single temperature scalar T > 0 to minimize calibration multi-class log-loss."""
    def loss_fn(temp: float) -> float:
        p = softmax(raw_logits / temp, axis=1)
        return multiclass_log_loss(y_true, p)

    res = minimize_scalar(loss_fn, bounds=(0.5, 3.0), method="bounded")
    return float(res.x)


val_logits_stats = lgb_1x2_stats.predict(cal_df[STATS_FEATURES], raw_score=True)
temp_stats = fit_temperature(val_logits_stats, cal_df["target_1x2"].values)

test_probs_1x2_raw = lgb_1x2_stats.predict_proba(test_df[STATS_FEATURES])
test_logits_stats = lgb_1x2_stats.predict(test_df[STATS_FEATURES], raw_score=True)
test_probs_1x2_cal = softmax(test_logits_stats / temp_stats, axis=1)

print(f"Pure-stats 1X2 best iter: {lgb_1x2_stats.best_iteration_} | Fitted temperature T = {temp_stats:.4f}")

# 6b. Stats + Market blended 3-class LightGBM (only rows with valid market odds)
train_mkt = train_df.dropna(subset=["mkt_p_home", "mkt_p_draw", "mkt_p_away"])
tune_mkt  = tune_df.dropna(subset=["mkt_p_home", "mkt_p_draw", "mkt_p_away"])
cal_mkt   = cal_df.dropna(subset=["mkt_p_home", "mkt_p_draw", "mkt_p_away"])

lgb_1x2_blend = lgb.LGBMClassifier(**CLF_PARAMS)
lgb_1x2_blend.fit(
    train_mkt[BLENDED_FEATURES], train_mkt["target_1x2"],
    eval_set=[(tune_mkt[BLENDED_FEATURES], tune_mkt["target_1x2"])],
    callbacks=[lgb.early_stopping(50, verbose=False)],
)
val_logits_blend = lgb_1x2_blend.predict(cal_mkt[BLENDED_FEATURES], raw_score=True)
temp_blend = fit_temperature(val_logits_blend, cal_mkt["target_1x2"].values)

test_logits_blend = lgb_1x2_blend.predict(test_df[BLENDED_FEATURES], raw_score=True)
test_probs_blend_cal = softmax(test_logits_blend / temp_blend, axis=1)
print(f"Blended 1X2 best iter:    {lgb_1x2_blend.best_iteration_} | Fitted temperature T = {temp_blend:.4f}")

## 7. PS-05 Diagnostics & Parametric Time-Decayed Dixon-Coles MLE

Merged from **`PS-05`**:
1. **Low-score correlation diagnostic:** compares empirical `0-0`, `1-0`, `0-1`, `1-1` frequencies against independent Poisson.
2. **Per-league Time-Decayed Dixon-Coles MLE:** fits `(attack_i, defense_i, home_adv, rho)` via `L-BFGS-B` with exponential time decay $w(t) = e^{-\xi \cdot \Delta\text{days}}$ and ridge regularization for newly promoted teams.
3. **Chronological $\xi$ tuning + Hybrid PS-05 × LightGBM ensemble:** tunes $\xi$ and the convex blend weight $\alpha$ strictly on the validation season (`val_df`), then evaluates out-of-sample on `test_df`.

In [ ]:
# 7a. PS-05 Low-Score Correlation Diagnostic (empirical vs independent Poisson)
mu_h_emp = train_df["home_goals"].mean()
mu_a_emp = train_df["away_goals"].mean()
low_scores = [(0, 0), (1, 0), (0, 1), (1, 1)]
diag_rows = []
for hg_v, ag_v in low_scores:
    obs = float(((train_df["home_goals"] == hg_v) & (train_df["away_goals"] == ag_v)).mean())
    exp = float(poisson.pmf(hg_v, mu_h_emp) * poisson.pmf(ag_v, mu_a_emp))
    diag_rows.append({
        "scoreline": f"{hg_v}-{ag_v}",
        "observed_freq": round(obs, 4),
        "indep_poisson": round(exp, 4),
        "ratio_obs_to_exp": round(obs / exp, 3) if exp > 0 else np.nan,
    })
print("PS-05 Low-Score Diagnostic (Train Set):")
display(pd.DataFrame(diag_rows))


def fit_league_dixon_coles_mle(
    league_df: pd.DataFrame,
    ref_date: pd.Timestamp,
    xi: float = 0.002,
    l2_reg: float = 1e-3,
    lookback_days: int = 1460,
) -> dict:
    """Fit parametric time-decayed Dixon-Coles MLE (attack, defense, home_adv, rho) for one league."""
    cutoff = ref_date - pd.Timedelta(days=lookback_days)
    sub = league_df[(league_df["match_date"] < ref_date) & (league_df["match_date"] >= cutoff)]
    if len(sub) < 80:
        sub = league_df[league_df["match_date"] < ref_date]

    teams = sorted(set(sub["home_team"]).union(set(sub["away_team"])))
    t2i = {t: idx for idx, t in enumerate(teams)}
    n_teams = len(teams)

    h_idx = sub["home_team"].map(t2i).values
    a_idx = sub["away_team"].map(t2i).values
    hg = sub["home_goals"].values.astype(np.int64)
    ag = sub["away_goals"].values.astype(np.int64)
    days_ago = np.maximum(0.0, (ref_date - sub["match_date"]).dt.days.values.astype(np.float64))
    weights = np.exp(-xi * days_ago)
    weights = weights / np.mean(weights)

    # Precompute low-score masks
    m00 = (hg == 0) & (ag == 0)
    m01 = (hg == 0) & (ag == 1)
    m10 = (hg == 1) & (ag == 0)
    m11 = (hg == 1) & (ag == 1)
    log_fact = np.array([math.lgamma(k + 1) for k in range(25)])
    const_term = log_fact[np.minimum(hg, 24)] + log_fact[np.minimum(ag, 24)]

    # Parameter vector: [attack (n), defense (n), home_adv (1), intercept (1), rho (1)]
    x0 = np.zeros(2 * n_teams + 3, dtype=np.float64)
    x0[2 * n_teams] = 0.25      # home_adv
    x0[2 * n_teams + 1] = 0.15  # intercept
    x0[2 * n_teams + 2] = -0.08 # rho

    bounds = (
        [(-2.0, 2.0)] * (2 * n_teams)
        + [(-0.5, 1.0), (-1.5, 1.5), (-0.30, 0.15)]
    )

    n_obs = len(hg)
    s_w = weights / n_obs

    def nll_and_grad(params: np.ndarray) -> tuple[float, np.ndarray]:
        att = params[:n_teams]
        def_ = params[n_teams:2 * n_teams]
        home_adv = params[2 * n_teams]
        intercept = params[2 * n_teams + 1]
        rho = params[2 * n_teams + 2]

        # Zero-sum centering for identifiability
        att_c = att - np.mean(att)
        def_c = def_ - np.mean(def_)

        log_lam = np.clip(intercept + home_adv + att_c[h_idx] - def_c[a_idx], -3.0, 2.2)
        log_mu  = np.clip(intercept + att_c[a_idx] - def_c[h_idx], -3.0, 2.2)
        lam = np.exp(log_lam)
        mu  = np.exp(log_mu)

        tau = np.ones(n_obs, dtype=np.float64)
        tau[m00] = 1.0 - lam[m00] * mu[m00] * rho
        tau[m01] = 1.0 + lam[m01] * rho
        tau[m10] = 1.0 + mu[m10] * rho
        tau[m11] = 1.0 - rho
        if np.any(tau <= 1e-6):
            return 1e9, np.zeros_like(params)

        ll = hg * log_lam - lam + ag * log_mu - mu - const_term + np.log(tau)
        reg = l2_reg * (np.sum(att_c ** 2) + np.sum(def_c ** 2))
        val = float(-np.sum(s_w * ll) + reg)

        # Exact analytical gradient (50x faster than finite differences)
        dtau_dloglam = np.zeros(n_obs, dtype=np.float64)
        dtau_dlogmu  = np.zeros(n_obs, dtype=np.float64)
        dtau_drho    = np.zeros(n_obs, dtype=np.float64)

        dtau_dloglam[m00] = (-lam[m00] * mu[m00] * rho) / tau[m00]
        dtau_dlogmu[m00]  = dtau_dloglam[m00]
        dtau_drho[m00]    = (-lam[m00] * mu[m00]) / tau[m00]

        dtau_dloglam[m01] = (lam[m01] * rho) / tau[m01]
        dtau_drho[m01]    = lam[m01] / tau[m01]

        dtau_dlogmu[m10]  = (mu[m10] * rho) / tau[m10]
        dtau_drho[m10]    = mu[m10] / tau[m10]

        dtau_drho[m11]    = -1.0 / tau[m11]

        g_loglam = -s_w * (hg - lam + dtau_dloglam)
        g_logmu  = -s_w * (ag - mu  + dtau_dlogmu)

        g_att_c = (
            np.bincount(h_idx, weights=g_loglam, minlength=n_teams)
            + np.bincount(a_idx, weights=g_logmu, minlength=n_teams)
            + 2.0 * l2_reg * att_c
        )
        g_def_c = (
            -np.bincount(a_idx, weights=g_loglam, minlength=n_teams)
            - np.bincount(h_idx, weights=g_logmu, minlength=n_teams)
            + 2.0 * l2_reg * def_c
        )

        g_att = g_att_c - np.mean(g_att_c)
        g_def = g_def_c - np.mean(g_def_c)
        g_home = float(np.sum(g_loglam))
        g_int  = float(np.sum(g_loglam + g_logmu))
        g_rho  = float(-np.sum(s_w * dtau_drho))

        grad = np.concatenate([g_att, g_def, [g_home, g_int, g_rho]])
        return val, grad

    res = minimize(nll_and_grad, x0, method="L-BFGS-B", jac=True, bounds=bounds, options={"maxiter": 250})
    p = res.x
    att = p[:n_teams] - np.mean(p[:n_teams])
    def_ = p[n_teams:2 * n_teams] - np.mean(p[n_teams:2 * n_teams])

    return {
        "home_adv": float(p[2 * n_teams]),
        "intercept": float(p[2 * n_teams + 1]),
        "rho": float(p[2 * n_teams + 2]),
        "xi": float(xi),
        "converged": bool(res.success),
        "teams": {
            t: {"attack": round(float(att[i]), 5), "defense": round(float(def_[i]), 5)}
            for i, t in enumerate(teams)
        },
    }


def predict_dixon_coles_mle(target_df: pd.DataFrame, league_models: dict[str, dict]) -> dict[str, np.ndarray]:
    """Predict lambdas and 1X2/Over2.5/BTTS probabilities from fitted per-league Dixon-Coles MLE params."""
    lam_h = np.full(len(target_df), 1.35, dtype=np.float64)
    lam_a = np.full(len(target_df), 1.10, dtype=np.float64)
    rhos  = np.full(len(target_df), DEFAULT_DC_RHO, dtype=np.float64)

    for i, r in enumerate(target_df.itertuples(index=False)):
        m = league_models.get(r.league_code)
        if not m:
            continue
        teams = m["teams"]
        # Newly promoted / unseen teams fall back to league-average (0.0, 0.0)
        h_att = teams.get(r.home_team, {}).get("attack", 0.0)
        h_def = teams.get(r.home_team, {}).get("defense", 0.0)
        a_att = teams.get(r.away_team, {}).get("attack", 0.0)
        a_def = teams.get(r.away_team, {}).get("defense", 0.0)

        lam_h[i] = math.exp(np.clip(m["intercept"] + m["home_adv"] + h_att - a_def, -3.0, 2.2))
        lam_a[i] = math.exp(np.clip(m["intercept"] + a_att - h_def, -3.0, 2.2))
        rhos[i]  = m["rho"]

    mkts = dixon_coles_markets(lam_h, lam_a, rho=rhos)
    mkts["lam_h"] = lam_h
    mkts["lam_a"] = lam_a
    return mkts


# 7b. Nested tuning: tune exponential time-decay xi strictly on tune_df (2022/23)
t_dc = time.time()
XI_GRID = [0.0, 0.001, 0.002, 0.0035]
tune_ref = pd.Timestamp(TRAIN_END_DATE)
best_xi = 0.002
best_xi_ll = float("inf")
xi_results = []

for xi_cand in XI_GRID:
    tune_models = {
        lc: fit_league_dixon_coles_mle(grp, ref_date=tune_ref, xi=xi_cand)
        for lc, grp in train_df.groupby("league_code")
    }
    tune_pred = predict_dixon_coles_mle(tune_df, tune_models)
    ll = multiclass_log_loss(tune_df["target_1x2"].values, tune_pred["probs_1x2"])
    br = multiclass_brier_score(tune_df["target_1x2"].values, tune_pred["probs_1x2"])
    xi_results.append({"xi": xi_cand, "tune_log_loss": round(ll, 5), "tune_brier": round(br, 5)})
    if ll < best_xi_ll:
        best_xi_ll = ll
        best_xi = xi_cand

print(f"PS-05 xi decay tuning (analytical grad) completed in {time.time() - t_dc:.1f}s -> best xi = {best_xi}")
display(pd.DataFrame(xi_results))

# 7c. Nested calibration: refit up to TUNE_END_DATE (train+tune) and calibrate strictly on cal_df (2023/24)
train_tune_df = pd.concat([train_df, tune_df], ignore_index=True)
cal_ref = pd.Timestamp(TUNE_END_DATE)
cal_dc_models = {
    lc: fit_league_dixon_coles_mle(grp, ref_date=cal_ref, xi=best_xi)
    for lc, grp in train_tune_df.groupby("league_code")
}
cal_dc_pred = predict_dixon_coles_mle(cal_df, cal_dc_models)

val_dc_logits = np.log(np.clip(cal_dc_pred["probs_1x2"], 1e-9, 1.0))
temp_dc_mle = fit_temperature(val_dc_logits, cal_df["target_1x2"].values)

# Refit up to CAL_END_DATE (train+tune+cal) for held-out test prediction
train_val_df = pd.concat([train_df, tune_df, cal_df], ignore_index=True)
test_ref = pd.Timestamp(CAL_END_DATE)
dc_mle_models = {
    lc: fit_league_dixon_coles_mle(grp, ref_date=test_ref, xi=best_xi)
    for lc, grp in train_val_df.groupby("league_code")
}
dc_mle_test = predict_dixon_coles_mle(test_df, dc_mle_models)
test_dc_mle_cal = softmax(np.log(np.clip(dc_mle_test["probs_1x2"], 1e-9, 1.0)) / temp_dc_mle, axis=1)

# 7d. Hybrid Ensemble: PS-05 Time-Decayed Dixon-Coles MLE + Dual LightGBM Poisson (tuned on cal_df)
val_lgb_dc = dixon_coles_markets(val_lam_h, val_lam_a, rho=DEFAULT_DC_RHO)["probs_1x2"]
cal_dc_cal = softmax(val_dc_logits / temp_dc_mle, axis=1)


def hybrid_loss(alpha: float) -> float:
    p = alpha * val_lgb_dc + (1.0 - alpha) * cal_dc_cal
    return multiclass_log_loss(cal_df["target_1x2"].values, p)


res_alpha = minimize_scalar(hybrid_loss, bounds=(0.0, 1.0), method="bounded")
best_hybrid_alpha = float(res_alpha.x)
test_hybrid_probs = best_hybrid_alpha * dc_fitted_test["probs_1x2"] + (1.0 - best_hybrid_alpha) * test_dc_mle_cal

print(f"PS-05 DC-MLE temperature T = {temp_dc_mle:.4f} | Hybrid alpha (LGBM weight) = {best_hybrid_alpha:.3f}")

## 8. Held-Out Test Season Benchmark, Clustered Bootstrap 95% CIs & Calibration Curves

Evaluated strictly on held-out matches after `VAL_END_DATE` (2024/25+ seasons), including **PS-05's matchday-clustered bootstrap 95% confidence intervals** (resampling by `match_date` so same-day correlations are preserved).

In [ ]:
y_test = test_df["target_1x2"].values
mkt_test_probs = test_df[["mkt_p_home", "mkt_p_draw", "mkt_p_away"]].values
has_odds = ~np.isnan(mkt_test_probs).any(axis=1)
print(f"Held-out test matches with closing odds: {has_odds.sum():,} / {len(test_df):,}")

model_preds = {
    "Bookmaker De-vigged Market (Baseline)": mkt_test_probs[has_odds],
    "PS-05 Time-Decayed Dixon-Coles MLE (cal)": test_dc_mle_cal[has_odds],
    "LGBM Poisson + Dixon-Coles (rho=-0.135)": dc_default_test["probs_1x2"][has_odds],
    "LGBM Poisson + Dixon-Coles (fitted rho)": dc_fitted_test["probs_1x2"][has_odds],
    "Hybrid (PS-05 DC-MLE + LGBM Poisson)": test_hybrid_probs[has_odds],
    "LGBM 1X2 Pure Stats (temp-calibrated)": test_probs_1x2_cal[has_odds],
    "LGBM 1X2 Stats + Market Blend (cal)": test_probs_blend_cal[has_odds],
}

results = [evaluate_1x2(name, y_test[has_odds], probs) for name, probs in model_preds.items()]
ps_test_probs = test_df[["ps_p_home", "ps_p_draw", "ps_p_away"]].values
has_ps = ~np.isnan(ps_test_probs).any(axis=1)
if has_ps.sum() > 100:
    results.append(evaluate_1x2("Pinnacle Closing Odds (PS-05 Sharp Baseline)", y_test[has_ps], ps_test_probs[has_ps]))
results_df = pd.DataFrame(results).sort_values("brier_3class").reset_index(drop=True)
display(results_df)

# PS-05 Matchday-Clustered Bootstrap 95% CIs (Brier & delta-Brier vs Bookmaker Market)
def clustered_bootstrap_ci(
    y_true: np.ndarray,
    probs_model: np.ndarray,
    probs_ref: np.ndarray,
    clusters: np.ndarray,
    n_boot: int = 500,
    seed: int = 42,
) -> dict:
    rng = np.random.default_rng(seed)
    y_oh = np.eye(3, dtype=np.float64)[y_true]
    b_mod = np.sum((probs_model - y_oh) ** 2, axis=1)
    b_ref = np.sum((probs_ref - y_oh) ** 2, axis=1)

    uniq_clusters, inv = np.unique(clusters, return_inverse=True)
    n_c = len(uniq_clusters)
    # Pre-aggregate sums & counts per matchday cluster for instant vectorized bootstrap
    sum_mod = np.bincount(inv, weights=b_mod, minlength=n_c)
    sum_ref = np.bincount(inv, weights=b_ref, minlength=n_c)
    cnt_c   = np.bincount(inv, minlength=n_c).astype(np.float64)

    boot_brier = np.empty(n_boot, dtype=np.float64)
    boot_diff  = np.empty(n_boot, dtype=np.float64)
    for b in range(n_boot):
        draw = rng.integers(0, n_c, size=n_c)
        tot_n = cnt_c[draw].sum()
        bm = sum_mod[draw].sum() / tot_n
        br = sum_ref[draw].sum() / tot_n
        boot_brier[b] = bm
        boot_diff[b]  = bm - br

    return {
        "brier_ci95_lo": round(float(np.percentile(boot_brier, 2.5)), 5),
        "brier_ci95_hi": round(float(np.percentile(boot_brier, 97.5)), 5),
        "delta_vs_mkt": round(float(np.mean(b_mod - b_ref)), 5),
        "delta_ci95_lo": round(float(np.percentile(boot_diff, 2.5)), 5),
        "delta_ci95_hi": round(float(np.percentile(boot_diff, 97.5)), 5),
    }


matchday_clusters = test_df.loc[has_odds, "match_date"].dt.strftime("%Y-%m-%d").values
ci_rows = []
for name, probs in model_preds.items():
    if name == "Bookmaker De-vigged Market (Baseline)":
        continue
    ci = clustered_bootstrap_ci(y_test[has_odds], probs, mkt_test_probs[has_odds], matchday_clusters)
    ci_rows.append({"model": name, **ci})

print("\nPS-05 Matchday-Clustered Bootstrap 95% CIs (vs Bookmaker Market):")
ci_df = pd.DataFrame(ci_rows).sort_values("delta_vs_mkt").reset_index(drop=True)
display(ci_df)

# Expanded Derived Markets Evaluation (Test Set)
has_ou = test_df["mkt_p_over25"].notna().values
derived_summary = [
    {"market": "Over 1.5 Goals", "lgbm_poisson_brier": round(binary_brier_score(test_df["target_over15"].values, dc_fitted_test["p_over15"]), 5), "ps05_dc_brier": round(binary_brier_score(test_df["target_over15"].values, dc_mle_test["p_over15"]), 5)},
    {"market": "Over 2.5 Goals (odds subset)", "lgbm_poisson_brier": round(binary_brier_score(test_df["target_over25"].values[has_ou], dc_fitted_test["p_over25"][has_ou]), 5), "ps05_dc_brier": round(binary_brier_score(test_df["target_over25"].values[has_ou], dc_mle_test["p_over25"][has_ou]), 5), "market_brier": round(binary_brier_score(test_df["target_over25"].values[has_ou], test_df["mkt_p_over25"].values[has_ou]), 5)},
    {"market": "Over 3.5 Goals", "lgbm_poisson_brier": round(binary_brier_score(test_df["target_over35"].values, dc_fitted_test["p_over35"]), 5), "ps05_dc_brier": round(binary_brier_score(test_df["target_over35"].values, dc_mle_test["p_over35"]), 5)},
    {"market": "BTTS (Both Teams to Score)", "lgbm_poisson_brier": round(binary_brier_score(test_df["target_btts"].values, dc_fitted_test["p_btts"]), 5), "ps05_dc_brier": round(binary_brier_score(test_df["target_btts"].values, dc_mle_test["p_btts"]), 5)},
    {"market": "Double Chance 1X", "lgbm_poisson_brier": round(binary_brier_score((y_test != 2).astype(int), dc_fitted_test["p_1x"]), 5), "ps05_dc_brier": round(binary_brier_score((y_test != 2).astype(int), dc_mle_test["p_1x"]), 5)},
    {"market": "Double Chance X2", "lgbm_poisson_brier": round(binary_brier_score((y_test != 0).astype(int), dc_fitted_test["p_x2"]), 5), "ps05_dc_brier": round(binary_brier_score((y_test != 0).astype(int), dc_mle_test["p_x2"]), 5)},
]
print("\nExpanded Derived Markets (Test Set):")
display(pd.DataFrame(derived_summary))

# ── v2.0 Prediction Confidence Engine (separate from raw probability!) ────────
# Combines: (1) model agreement across LGBM Poisson, LGBM 1X2, and PS-05 DC-MLE,
#           (2) data depth (team matches played + league reliability tier),
#           (3) historical bucket calibration error measured on cal_df.
cal_curve_ref = calibration_curve_punterstat(cal_df["target_1x2"].values, val_lgb_dc)
bucket_cal_err = {
    row["bucket"]: abs(float(row["predicted_avg"]) - float(row["actual_freq"]))
    if pd.notna(row["predicted_avg"]) else 0.05
    for _, row in cal_curve_ref.iterrows()
}

def _bucket_err_for_p(p_val: float) -> float:
    if p_val < 0.2: return bucket_cal_err.get("0-20%", 0.03)
    if p_val < 0.4: return bucket_cal_err.get("20-40%", 0.03)
    if p_val < 0.6: return bucket_cal_err.get("40-60%", 0.03)
    if p_val < 0.8: return bucket_cal_err.get("60-80%", 0.03)
    return bucket_cal_err.get("80-100%", 0.03)

ens_stack = np.stack([dc_fitted_test["probs_1x2"], test_probs_1x2_cal, test_dc_mle_cal], axis=0)
model_disagreement = np.mean(np.std(ens_stack, axis=0), axis=1)  # lower = stronger agreement
min_history = np.minimum(test_df["h_matches_played"].values, test_df["a_matches_played"].values)
tier_score = test_df["league_code"].map(LEAGUE_RELIABILITY).map({"Gold": 1.0, "Silver": 0.7, "Bronze": 0.4}).fillna(0.4).values
top_p = np.max(dc_fitted_test["probs_1x2"], axis=1)
cal_pen = np.array([_bucket_err_for_p(p) for p in top_p])

conf_score = (
    0.45 * np.clip(1.0 - model_disagreement / 0.08, 0.0, 1.0)
    + 0.35 * (0.5 * np.clip(min_history / 20.0, 0.0, 1.0) + 0.5 * tier_score)
    + 0.20 * np.clip(1.0 - cal_pen / 0.06, 0.0, 1.0)
)
conf_tier = np.where(conf_score >= 0.75, "High", np.where(conf_score >= 0.55, "Medium", "Low"))

conf_eval_rows = []
for tier in ["High", "Medium", "Low"]:
    m_t = (conf_tier == tier) & has_odds
    if m_t.sum() > 20:
        r_t = evaluate_1x2(f"Confidence: {tier}", y_test[m_t], dc_fitted_test["probs_1x2"][m_t])
        r_t["avg_disagreement"] = round(float(model_disagreement[m_t].mean()), 4)
        conf_eval_rows.append(r_t)
print("\nv2.0 Prediction Confidence Engine Evaluation (LGBM Poisson+DC on Test Set):")
display(pd.DataFrame(conf_eval_rows))

# ── Per-League Test Evaluation (Top 15 Leagues by Test Volume) ────────────────
per_league_rows = []
for lc, grp in test_df.groupby("league_code"):
    pos = test_df.index.get_indexer(grp.index)
    m_lc = has_odds[pos]
    if m_lc.sum() < 50:
        continue
    pos_o = pos[m_lc]
    b_mod = multiclass_brier_score(y_test[pos_o], dc_fitted_test["probs_1x2"][pos_o])
    b_mkt = multiclass_brier_score(y_test[pos_o], mkt_test_probs[pos_o])
    per_league_rows.append({
        "league_code": lc,
        "tier": LEAGUE_RELIABILITY.get(lc, "Bronze"),
        "n_test": int(len(pos_o)),
        "lgbm_brier": round(b_mod, 5),
        "mkt_brier": round(b_mkt, 5),
        "delta_vs_mkt": round(b_mod - b_mkt, 5),
    })
per_league_df = pd.DataFrame(per_league_rows).sort_values("lgbm_brier").reset_index(drop=True)
print("\nPer-League Test Brier Breakdown (best 15 leagues):")
display(per_league_df.head(15))

# ── 3-Fold Walk-Forward Multi-Season Validation (LGBM Poisson + DC) ───────────
wf_folds = [
    ("2022/23", "2021-07-01", "2022-07-01", "2023-07-01"),
    ("2023/24", "2022-07-01", "2023-07-01", "2024-07-01"),
    ("2024/25+", "2023-07-01", "2024-07-01", "2027-07-01"),
]
wf_rows = []
for fold_name, tr_end, va_end, te_end in wf_folds:
    f_tr = data[data["match_date"] < tr_end]
    f_va = data[(data["match_date"] >= tr_end) & (data["match_date"] < va_end)]
    f_te = data[(data["match_date"] >= va_end) & (data["match_date"] < te_end)]
    if len(f_te) < 500:
        continue
    mh = lgb.LGBMRegressor(**{**POISSON_PARAMS, "n_estimators": 800})
    mh.fit(f_tr[STATS_FEATURES], f_tr["home_goals"], eval_set=[(f_va[STATS_FEATURES], f_va["home_goals"])], callbacks=[lgb.early_stopping(40, verbose=False)])
    ma = lgb.LGBMRegressor(**{**POISSON_PARAMS, "n_estimators": 800})
    ma.fit(f_tr[STATS_FEATURES], f_tr["away_goals"], eval_set=[(f_va[STATS_FEATURES], f_va["away_goals"])], callbacks=[lgb.early_stopping(40, verbose=False)])
    f_probs = dixon_coles_markets(mh.predict(f_te[STATS_FEATURES]), ma.predict(f_te[STATS_FEATURES]), rho=global_rho)["probs_1x2"]
    f_mkt = f_te[["mkt_p_home", "mkt_p_draw", "mkt_p_away"]].values
    f_ok = ~np.isnan(f_mkt).any(axis=1)
    wf_rows.append({
        "test_window": fold_name,
        "n_matches": int(f_ok.sum()),
        "lgbm_brier": round(multiclass_brier_score(f_te["target_1x2"].values[f_ok], f_probs[f_ok]), 5),
        "mkt_brier": round(multiclass_brier_score(f_te["target_1x2"].values[f_ok], f_mkt[f_ok]), 5),
        "lgbm_log_loss": round(multiclass_log_loss(f_te["target_1x2"].values[f_ok], f_probs[f_ok]), 5),
    })
print("\n3-Fold Walk-Forward Multi-Season Validation:")
display(pd.DataFrame(wf_rows))

# Reliability / Calibration Curve + Feature Importance Plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot([0, 1], [0, 1], "k--", alpha=0.6, label="Perfect Calibration")
for label, probs in [
    ("Bookmaker Market", mkt_test_probs[has_odds]),
    ("Hybrid (PS-05 + LGBM)", test_hybrid_probs[has_odds]),
    ("PS-05 DC-MLE (cal)", test_dc_mle_cal[has_odds]),
    ("LGBM Poisson + DC (fitted)", dc_fitted_test["probs_1x2"][has_odds]),
]:
    curve = calibration_curve_punterstat(y_test[has_odds], probs)
    ax.plot(curve["predicted_avg"], curve["actual_freq"], marker="o", linewidth=2, label=label)

ax.set_title("PunterStat v2.0 5-Bucket Calibration Curve (Test Season)")
ax.set_xlabel("Predicted Probability")
ax.set_ylabel("Observed Outcome Frequency")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)
ax.legend()

imp = pd.Series(lgb_home.feature_importances_, index=STATS_FEATURES).sort_values(ascending=True).tail(15)
axes[1].barh(imp.index, imp.values, color="#2563eb")
axes[1].set_title("Top 15 Features — Home Expected Goals (v2.0 LightGBM Poisson)")
axes[1].set_xlabel("Split Gain / Count Importance")
axes[1].grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Export v2.0 Model Registry Artifacts for PunterStat (Vercel / Supabase Ready)

Saves versioned artifacts to `/kaggle/working/punterstat_artifacts/`:
- `dixon_coles_mle_params.json` — lightweight per-league `(attack, defense, home_adv, intercept, rho)` JSON for native TypeScript inference on Vercel.
- `model_metadata.json` — **v2.0 Model Registry** manifest (`model_version`, `feature_schema_version`, `dataset_sha256`, `trained_at_utc`, walk-forward validation, per-league metrics, confidence-tier metrics, reliability tiers, and clustered bootstrap 95% CIs).
- LightGBM `.txt` models (`lgb_home_goals.txt`, `lgb_away_goals.txt`, `lgb_1x2_stats.txt`, `lgb_1x2_blend.txt`).

In [ ]:
EXPORT_DIR = WORK_DIR / "punterstat_artifacts"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

lgb_home.booster_.save_model(str(EXPORT_DIR / "lgb_home_goals.txt"))
lgb_away.booster_.save_model(str(EXPORT_DIR / "lgb_away_goals.txt"))
lgb_1x2_stats.booster_.save_model(str(EXPORT_DIR / "lgb_1x2_stats.txt"))
lgb_1x2_blend.booster_.save_model(str(EXPORT_DIR / "lgb_1x2_blend.txt"))

with open(EXPORT_DIR / "dixon_coles_mle_params.json", "w", encoding="utf-8") as f:
    json.dump(dc_mle_models, f, indent=2)

metadata = {
    "model_version": MODEL_VERSION,
    "feature_schema_version": FEATURE_SCHEMA_VERSION,
    "dataset_sha256": DATASET_SHA256,
    "trained_at_utc": dt_mod.datetime.now(dt_mod.timezone.utc).isoformat(),
    "splits": {
        "train_end": TRAIN_END_DATE,
        "tune_end": TUNE_END_DATE,
        "cal_end": CAL_END_DATE,
    },
    "best_dc_mle_xi": best_xi,
    "temperature_dc_mle": round(temp_dc_mle, 4),
    "hybrid_lgbm_weight_alpha": round(best_hybrid_alpha, 4),
    "global_dc_rho": round(global_rho, 4),
    "league_dc_rho": league_rhos,
    "league_reliability": LEAGUE_RELIABILITY,
    "temperature_1x2_stats": round(temp_stats, 4),
    "temperature_1x2_blend": round(temp_blend, 4),
    "stats_features": STATS_FEATURES,
    "blended_features": BLENDED_FEATURES,
    "test_benchmark": results,
    "derived_markets_benchmark": derived_summary,
    "confidence_tier_evaluation": conf_eval_rows,
    "walk_forward_validation": wf_rows,
    "per_league_test_brier": per_league_rows,
    "clustered_bootstrap_ci": ci_rows,
}

with open(EXPORT_DIR / "model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"Exported PunterStat Model Registry v{MODEL_VERSION} artifacts to: {EXPORT_DIR}")
for p in sorted(EXPORT_DIR.iterdir()):
    print(f"  - {p.name} ({p.stat().st_size / 1024:.1f} KB)")

print("\nReady-to-paste TypeScript map for analyzer.ts:")
print("export const LEAGUE_DC_RHO: Record<string, number> =", json.dumps(league_rhos, indent=2), ";")